## EDA, Sistema de Predicción Energética y Alerta Temprana en Barrios de Barcelona

> **Contexto:** Dataset con consumo eléctrico por código postal (bloques de 6h, 2019–2025), enriquecido con variables meteorológicas de estaciones Meteocat y temperatura superficial satelital (MODIS LST). El objetivo del EDA es comprender la estructura, calidad y comportamiento del dataset antes del modelado: identificar patrones temporales, relaciones entre variables y anomalías que condicionen las decisiones de feature engineering y arquitectura del modelo.

### <font color='#C0392B'><b>Variables del dataset</b></font>

| Variable | Tipo | Descripción |
|---|---|---|
| `cod_postal` | Categórica | Código postal de Barcelona (08001–08042) |
| `nombre_postal` | Categórica | Nombre del barrio asociado al código postal |
| `centroide_lon` | Numérica | Longitud del centroide del código postal |
| `centroide_lat` | Numérica | Latitud del centroide del código postal |
| `codi_estacio` | Categórica | Código de estación Meteocat asignada |
| `nombre_estacio` | Categórica | Nombre de la estación Meteocat asignada |
| `estacio_lon` | Numérica | Longitud de la estación Meteocat asignada |
| `estacio_lat` | Numérica | Latitud de la estación Meteocat asignada |
| `datetime` | Temporal | Inicio del bloque de 6 horas |
| `mwh_total` | Target | Consumo eléctrico total en MWh |
| `mwh_industria` | Numérica | Consumo sector industria |
| `mwh_residencial` | Numérica | Consumo sector residencial |
| `mwh_servicios` | Numérica | Consumo sector servicios |
| `mwh_no_especificado` | Numérica | Consumo no clasificado |
| `lst_celsius` | Numérica | Land Surface Temperature — MODIS (°C) |
| `temp_mean` | Numérica | Temperatura media Meteocat (°C) |
| `temp_max` | Numérica | Temperatura máxima Meteocat (°C) |
| `temp_min` | Numérica | Temperatura mínima Meteocat (°C) |
| `humedad_mean` | Numérica | Humedad relativa media (%) |
| `viento_mean` | Numérica | Velocidad media del viento (m/s) |
| `precipitacion_sum` | Numérica | Precipitación acumulada en el bloque (mm) |
| `irradiancia_mean` | Numérica | Irradiancia solar global media (W/m²) |
| `es_festivo` | Binaria | Es día festivo |
| `nombre_local` | Categórica | Nombre del festivo local |
| `hora` | Ordinal | Hora de inicio del bloque (0, 6, 12, 18) |
| `dia_semana` | Ordinal | Día de la semana (0=Lunes, 6=Domingo) |
| `mes` | Ordinal | Mes del año (1–12) |
| `anio` | Numérica | Año |
| `semana_anio` | Numérica | Semana del año (1–53) |
| `es_finde` | Binaria | Es fin de semana |

## <font color='#1B3A5C'>  **Librerías** </font>

In [ ]:
import polars as pl
import polars.selectors as cs
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import numpy as np
import scipy.stats as stats
from scipy.stats import kruskal, mannwhitneyu
from scipy.signal import correlate
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from pymongo import MongoClient
from statsmodels.tsa.seasonal import STL
from arch.unitroot import ADF, KPSS
from datetime import date
import json
import plotly.express as px
import pandas as pd
import time
from arch.unitroot import ADF, KPSS


plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#D3D3D3'
plt.rcParams['grid.linewidth'] = 0.4
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.xmargin'] = 0.01
plt.rcParams['axes.ymargin'] = 0.01

# Paleta de color
C1 = '#264653'    # azul oscuro
C2 = '#2A9D8F'    # verde azulado
C3 = '#E9C46A'    # amarillo dorado
C4 = '#F4A261'    # naranja
C5 = '#E76F51'    # rojo-naranja
C6 = '#A8DADC'    # azul claro
TITULO    = '#1B3A5C'  # azul marino — títulos de sección
SUBTITULO = '#C0392B'  # rojo oscuro — subtítulos

# <font color='#1B3A5C'>  **1. Carga y Exploración** </font>

### <font color='#C0392B'><b>1.1 Exploración Inicial</b></font>


In [ ]:
start_time = time.time()

In [ ]:
client = MongoClient('mongodb://mongo:27017/')
db     = client['tfm_energy']

docs = list(db['dataset_eda'].find({}, {'_id': 0}))
df   = pl.DataFrame(docs, infer_schema_length=None)

In [ ]:
df.head(6)

In [ ]:
df.tail(6)

In [ ]:
df.filter(pl.col("codi_estacio") == "X2").select(pl.col(["temp_mean", "temp_max", "temp_min", "humedad_mean", "viento_mean", "precipitacion_sum"])).head(6)

In [ ]:
print(f"Shape: {df.shape}")
print(f"Desde: {df['datetime'].min()}")
print(f"Hasta: {df['datetime'].max()}")
print(f"Códigos postales únicos: {df['cod_postal'].n_unique()}")
print(f"Años cubiertos: {sorted(df['anio'].unique().to_list())}")

In [ ]:
# Tipos de datos
print("SCHEMA")
for col, dtype in zip(df.columns, df.dtypes):
    print(f"  {col:<30} {dtype}")

In [ ]:
pl.Config.set_tbl_rows(50)

nulls = (
    pl.DataFrame({
        'columna': df.columns,
        'nulos':   [df[c].null_count() for c in df.columns]
    })
    .with_columns((pl.col('nulos') / len(df) * 100).round(2).alias('pct_nulos'))
    .sort('nulos', descending=True)
    .filter(pl.col('nulos') > 0)
)

print(nulls)

> El dataset cubre 424.148 registros en 42 códigos postales de Barcelona, bloques de 6h desde enero 2019 hasta junio 2025. 

- Cada fila representa un único bloque por código postal 
(datetime + cod_postal` como clave compuesta), agrupando variables de consumo eléctrico, meteorología Meteocat y temperatura superficial MODIS LST.

- El análisis de nulos identifica tres grupos: 

    - nombre_local (96%) y mwh_no_especificado (85%). Nombre local es el nombre de la festividad asíq eu teine sentido y mwh no especificado es porque es un valor añadido en 2025 para el dataset
    - Las variables  meteorológicas (32–36%) 
    - LST MODIS (39.8%) refleja cobertura nubosa y actúa como respaldo espacial.

EL resto deberán ser revisados

### <font color='#C0392B'><b>1.2 Distribución Inicial de datos faltantes</b></font>


In [ ]:
codpos_null = (
df.group_by('anio').agg([
    pl.col('irradiancia_mean').is_null().sum().alias('nulos_irradiancia'),
    pl.col('viento_mean').is_null().sum().alias('nulos_viento'),
    pl.col('temp_mean').is_null().sum().alias('nulos_temp'),
    pl.col('humedad_mean').is_null().sum().alias('nulos_humedad'),
    pl.len().alias('total_registros')
]).sort('anio'))

print(codpos_null)

In [ ]:
codpos_null = (df.filter(pl.col('viento_mean').is_null())
    .group_by(['cod_postal','nombre_postal','codi_estacio'])
    .agg(pl.len().alias('nulos_viento'))
    .sort('nulos_viento', descending=True)
    .select(['cod_postal','nombre_postal', 'nulos_viento','codi_estacio']))

print(codpos_null)

> Los nulos de viento e irradiancia son constantes desde 2019, la estación X2 no dispone de estos sensores para la mayoría de códigos postales, lo que explica los 22.000 nulos constantes por año. Los códigos postales asignados a D5, X4 y X8 presentan nulos esporádicos, confirmando que el problema es exclusivo de X2. La estación AN, con 10.100 nulos está totalmente vacía

> En 2024 y 2025 los nulos de temperatura y humedad escalan hasta igualarse con los de irradiancia y viento aprox 20.249, confirmando  la inactividad progresiva de X2, que en 2025 deja de reportar prácticamente todas las variables.

### <font color='#C0392B'><b>1.3 Cobertura de estaciones de meteocat Barcelona</b></font>


In [ ]:
df.group_by(["codi_estacio", "anio"]).agg([
    pl.len().alias("total_registros"),
    pl.col("temp_mean").is_null().sum().alias("nulos_temp"),
    pl.col("viento_mean").is_null().sum().alias("nulos_viento"),
    pl.col("irradiancia_mean").is_null().sum().alias("nulos_irradiancia"),
]).sort(["codi_estacio", "anio"])

In [ ]:
(
    df.group_by("codi_estacio")
    .agg([
        pl.len().alias("total"),
        (pl.col("temp_mean").is_null().sum() / pl.len() * 100).round(2).alias("pct_temp"),
        (pl.col("humedad_mean").is_null().sum() / pl.len() * 100).round(2).alias("pct_humedad"),
        (pl.col("viento_mean").is_null().sum() / pl.len() * 100).round(2).alias("pct_viento"),
        (pl.col("irradiancia_mean").is_null().sum() / pl.len() * 100).round(2).alias("pct_irradiancia"),
        (pl.col("lst_celsius").is_null().sum() / pl.len() * 100).round(2).alias("pct_lst"),
    ])
    .sort("codi_estacio")
)

- X4 : 0.57% nulos 
- X8 : 0.62% nulos 
- D5 : 0.80% nulos  
- X2 : 17.23% nulos 
- AN : 100% nulos

> **Estación AN, Parc de la Ciutadella:** AN y X2 se solapan físicamente, a menos de 200 metros dentro del mismo parque. Más importante aún, AN tiene el 100% de nulos en todas sus variables meteorológicas en todos los años, de 2019 a 2025, , así que AN existe como punto de referencia geográfico en Meteocat pero no dispone de sensores activos. El paso a seguir va a ser reasignar todos los registros originalmente asignados a AN hacia X2, que sí tiene datos de temperatura.

> **Estación X2, Zoo de Barcelona:** X2 presenta dos limitaciones. 
    - Primero, nunca ha tenido sensores de viento ni irradiancia ya que ambas variables tienen el 100% de nulos en todos los años. 
    - Segundo, a partir de 2024 comienza a fallar y en 2025 deja de funcionar completamente, con el 100% de nulos en temperatura. Para los códigos postales asignados a X2, viento e irradiancia se imputarán desde X4 en feature engineering; temperatura desde X4 con factor de corrección histórico ratio X2/X4 (2019–2023).

# <font color='#1B3A5C'>  **2. Limpieza inicial** </font>

> A partir de los hallazgos anteriores comenzaré con una limpieza inicial

### <font color='#C0392B'><b>2.1 Trato de Datos Faltantes</b></font>

#### <font color='#2A9D8F'><b>2.1.1 Datos sin registros en mwh_total</b></font>

In [ ]:
mwh_problematicos = df.filter(pl.col('mwh_total') <= 0)
print(f"Registros con mwh_total mayor o igual a 0: {len(mwh_problematicos):,}")

n_dupes = len(df) - df.unique(subset=['datetime', 'cod_postal']).shape[0]
print(f"Duplicados datetime + cod_postal: {n_dupes:,}")

> - Existen 378 filas donde no existen valores iguales o menores a 0 en el mwh_total lo cual se puede tratar de un problema de recolección de datos

In [ ]:
print(mwh_problematicos.select([
    "cod_postal", "nombre_postal", "datetime",
    "mwh_total", "mwh_industria", "mwh_residencial",
    "mwh_servicios", "mwh_no_especificado"
]))

In [ ]:
print(mwh_problematicos.select("datetime").unique().sort("datetime"))

In [ ]:
fechas_problema = ["2025-06-26", "2025-08-30", "2025-09-07"]

df.filter(
    (pl.col("datetime").dt.date().cast(pl.Utf8).is_in(fechas_problema)) &
    (pl.col("datetime").dt.hour() == 18)
).select(["cod_postal", "datetime", "mwh_total"]).head(10)

In [ ]:
mwh_problematicos.group_by("datetime") \
    .agg(pl.n_unique("cod_postal").alias("n_codigos")) \
    .sort("datetime")

> - Este error en loso datos existe durante todos lo días para los 42 códigos postales

> - Se detectaron 378 registros con mwh_total = 0, concentrados en exactamente 3 fechas de 2025 (26-jun, 30-ago, 07-sep). En cada caso, los bloques 00:00,  06:00 y 12:00 de los 42 códigos postales reportaron cero simultáneamente, mientras el bloque 18:00 del mismo día registró valores normales. 

> - Es un fallo parcial de OpenDataBCN, no con un corte real de suministro.

### <font color='#C0392B'><b>Verificar continuidad temporal</b></font>

In [ ]:
fechas_esperadas = df.select('cod_postal').unique().join(
    pl.datetime_range(df['datetime'].min(), df['datetime'].max(), interval='6h', eager=True)
      .alias('datetime').to_frame(), how='cross'
)
faltantes = fechas_esperadas.join(df.select(['cod_postal', 'datetime']), on=['cod_postal', 'datetime'], how='anti')
print(f"Faltantes detectados: {len(faltantes):,}")
print(faltantes.filter(pl.col('datetime').dt.date() != date(2025, 8, 19))
               .group_by('cod_postal').agg(pl.len().alias('n')).sort('n', descending=True))

docs_festivos = list(db["clean_festivos"].find({}, {"_id": 0}))
fechas_festivas = set(
    pl.DataFrame(docs_festivos)
    .with_columns(pl.col("fecha").str.to_date("%Y-%m-%d"))["fecha"].to_list()
)

referencia = (
    df.filter(pl.col("anio") < 2025)
    .group_by(["cod_postal", "hora", "dia_semana", "mes"])
    .agg(pl.col("mwh_total").median().alias("mwh_total"))
)

# Columnas estáticas por cod_postal (nombre, coordenadas, estación)
estaticas = (
    df.filter(pl.col("nombre_postal").is_not_null())
    .select(["cod_postal", "nombre_postal", "centroide_lon", "centroide_lat",
             "codi_estacio", "nombre_estacio", "estacio_lon", "estacio_lat"])
    .unique(subset=["cod_postal"])
)

df_faltantes = (
    faltantes
    .with_columns([
        pl.col("datetime").dt.hour().cast(pl.Int8).alias("hora"),
        pl.col("datetime").dt.weekday().cast(pl.Int8).alias("dia_semana"),
        pl.col("datetime").dt.month().cast(pl.Int8).alias("mes"),
    ])
    .join(referencia, on=["cod_postal", "hora", "dia_semana", "mes"], how="left")
    .join(estaticas, on="cod_postal", how="left")  # <-- fix: añade columnas estáticas
    .with_columns(
        pl.col("datetime").dt.date()
          .map_elements(lambda d: 1 if d in fechas_festivas else 0, return_dtype=pl.Int64)
          .alias("es_festivo")
    )
)

df = pl.concat([df, df_faltantes], how="diagonal_relaxed").sort(["cod_postal", "hora", "datetime"])
print(f"Shape: {df.shape} | Nulls mwh_total: {df['mwh_total'].null_count()}")
print(f"Nulls nombre_postal: {df['nombre_postal'].null_count()}")  # debe ser 0

In [ ]:
faltantes_por_dia = faltantes.with_columns(
    pl.col("datetime").dt.date().alias("fecha")
).group_by(["cod_postal", "fecha"]).count()

faltantes_por_dia.filter(
    pl.col("count") != 4  # 4 registros por día (cada 6h)
)

> - A todos estos días les faltan 4 de los 4 horarios

- 08011 (7–20 ago 2025): 52 bloques faltantes (13 días × 4). Problema de reporte específico del CP.
- 2025-08-19: día completo sin datos para los 42 CPs (168 bloques). Fallo de reporte en la fuente.

### <font color='#C0392B'><b>Conversión de tipo de datos</b></font>

In [ ]:
df = df.with_columns([
    pl.col('datetime').cast(pl.Datetime),
    pl.col('cod_postal').cast(pl.Utf8),
    pl.col('es_festivo').cast(pl.Int8),
    pl.col('es_finde').cast(pl.Int8),
    pl.col('hora').cast(pl.Int8),
    pl.col('dia_semana').cast(pl.Int8),
    pl.col('mes').cast(pl.Int8),
    pl.col('anio').cast(pl.Int16),
])
df.head(3)

### <font color='#C0392B'><b>Trato de Nulls</b></font>

1. **Reasignación AN X2**, variables meteorológicas rellenadas con datos reales de X2 desde MongoDB  
2. **Grilla temporal completa**, bloques faltantes (08011 ago 2025 + 19-ago global) rellenados con mediana histórica  
3. **Sectores de consumo**, mwh_industria, mwh_residencial y mwh_servicios imputados a 0 (sector no activo ese bloque); se recalcula mwh_total  
4. **Días con mwh_total = 0**, 3 fechas de fallo de reporte imputadas con mediana histórica 2019-2024  
5. **Nulls meteorológicos restantes**, viento, irradiancia, temp, humedad: estructurales, se mantienen para feature engineering

#### Estrategia de nulos meteorológicos, decisión EDA

>**Variables: viento_mean, irradiancia_mean, precipitacion_sum**  

X2 no mide estas variables en ningún período.
- Mantener nulos. Si el EDA muestra diferencias significativas entre barrios, para feature engineering

>**Variables: temp_mean, humedad_mean, período 2024 parcial y 2025**  
X2 inactiva. Datos reales disponibles para 2019–2023.
- Mantener nulos por ahora. Si el EDA confirma diferencias de comportamiento entre barrios (consumo vs temperatura), aplicar factor de corrección basado en la relación histórica X2/X4 de años anteriores.

>**Criterio de decisión (post-EDA):**
- Si variación entre barrios es baja: nulos sin imputar, el modelo los gestiona
- Si variación entre barrios es alta: factor de corrección histórico X2/X4

### 1. **Reasignación AN X2**

In [ ]:
df = df.with_columns([
    pl.when(pl.col("codi_estacio") == "AN").then(pl.lit("X2")).otherwise(pl.col("codi_estacio")).alias("codi_estacio"),
    pl.when(pl.col("codi_estacio") == "AN").then(pl.lit("Barcelona - Zoo")).otherwise(pl.col("nombre_estacio")).alias("nombre_estacio"),
    pl.when(pl.col("codi_estacio") == "AN").then(pl.lit(2.1898)).otherwise(pl.col("estacio_lon")).alias("estacio_lon"),
    pl.when(pl.col("codi_estacio") == "AN").then(pl.lit(41.3842)).otherwise(pl.col("estacio_lat")).alias("estacio_lat"),
])

docs_meteocat = list(db["clean_meteocat"].find({"codi_estacio": "X2"}, {"_id": 0}))
meteocat_x2 = (
    pl.DataFrame(docs_meteocat)
    .rename({"data_lectura": "datetime", "temp": "temp_mean", "humedad": "humedad_mean",
             "viento": "viento_mean", "precipitacion": "precipitacion_sum", "irradiancia": "irradiancia_mean"})
    .select(["datetime", "temp_mean", "humedad_mean", "viento_mean", "precipitacion_sum", "irradiancia_mean"])
)

df = df.join(meteocat_x2, on="datetime", how="left", suffix="_x2")

cols_meteo = ["temp_mean", "humedad_mean", "viento_mean", "precipitacion_sum", "irradiancia_mean"]
df = df.with_columns([
    pl.when(pl.col(c).is_null()).then(pl.col(f"{c}_x2")).otherwise(pl.col(c)).alias(c)
    for c in cols_meteo
]).drop([f"{c}_x2" for c in cols_meteo])

print("Reasignación AN a X2 completada")
print(df.group_by("codi_estacio").agg(pl.col("temp_mean").null_count().alias("nulos_temp"), pl.len().alias("total")).sort("codi_estacio"))

#### 2. Grilla temporal completa

08011 tiene 52 bloques faltantes en ago 2025 y el 19-ago-2025 falta para todos los CPs. Imputado con mediana histórica por CP + hora + dia_semana + mes (2019-2024).

In [ ]:
fechas_esperadas = df.select('cod_postal').unique().join(
    pl.datetime_range(df['datetime'].min(), df['datetime'].max(), interval='6h', eager=True)
      .alias('datetime').to_frame(), how='cross'
)
faltantes = fechas_esperadas.join(df.select(['cod_postal', 'datetime']), on=['cod_postal', 'datetime'], how='anti')
print(f"Faltantes detectados: {len(faltantes):,}")
print(faltantes.filter(pl.col('datetime').dt.date() != date(2025, 8, 19))
               .group_by('cod_postal').agg(pl.len().alias('n')).sort('n', descending=True))

docs_festivos = list(db["clean_festivos"].find({}, {"_id": 0}))
fechas_festivas = set(
    pl.DataFrame(docs_festivos)
    .with_columns(pl.col("fecha").str.to_date("%Y-%m-%d"))["fecha"].to_list()
)

referencia = (
    df.filter(pl.col("anio") < 2025)
    .group_by(["cod_postal", "hora", "dia_semana", "mes"])
    .agg(pl.col("mwh_total").median().alias("mwh_total"))
)

df_faltantes = (
    faltantes
    .with_columns([
        pl.col("datetime").dt.hour().cast(pl.Int8).alias("hora"),
        pl.col("datetime").dt.weekday().cast(pl.Int8).alias("dia_semana"),
        pl.col("datetime").dt.month().cast(pl.Int8).alias("mes"),
    ])
    .join(referencia, on=["cod_postal", "hora", "dia_semana", "mes"], how="left")
    .with_columns(
        pl.col("datetime").dt.date()
          .map_elements(lambda d: 1 if d in fechas_festivas else 0, return_dtype=pl.Int64)
          .alias("es_festivo")
    )
)

df = pl.concat([df, df_faltantes], how="diagonal_relaxed").sort(["cod_postal", "hora", "datetime"])
print(f"Shape: {df.shape} | Nulls mwh_total: {df['mwh_total'].null_count()}")

>- **08011 (7–20 ago 2025):** 52 bloques faltantes (13 días × 4). Problema de reporte específico del CP.

>- **2025-08-19:** día completo sin datos para los 42 CPs (168 bloques). Fallo OpenDataBCN.

>- Ambos se imputan con mediana histórica por CP + hora + día semana + mes, usando 2019-2024.

#### 3. Fill null sectores + recálculo mwh_total

Sectores sin dato = 0 (sector no activo ese bloque horario)

In [ ]:
df = df.with_columns([
    pl.col("mwh_industria").fill_null(0),
    pl.col("mwh_residencial").fill_null(0),
    pl.col("mwh_servicios").fill_null(0),
])

df = df.with_columns(
    pl.when(pl.col("mwh_total").is_null())
    .then(
        pl.col("mwh_industria") + pl.col("mwh_residencial") +
        pl.col("mwh_servicios") + pl.col("mwh_no_especificado").fill_null(0)
    )
    .otherwise(pl.col("mwh_total"))
    .alias("mwh_total")
)
# Verificar nulos restantes
nulos = (
    df.null_count()
    .unpivot(variable_name="columna", value_name="nulos")
    .filter(pl.col("nulos") > 0)
    .with_columns((pl.col("nulos") / len(df) * 100).round(2).alias("pct"))
    .sort("nulos", descending=True)
)
print(f"Shape: {df.shape}")
print(nulos)

#### 4. Imputar mwh_total = 0 (3 fechas de fallo de reporte en 2025)

26-jun, 30-ago y 07-sep 2025: mwh_total = 0 para los 42 CPs por error en OpenData. Se imputa con mediana histórica por CP + hora + dia_semana + mes (2019-2024).

In [ ]:
referencia_ceros = (
    df.filter((pl.col("anio") < 2025) & (pl.col("mwh_total") > 0))
    .group_by(["cod_postal", "hora", "dia_semana", "mes"])
    .agg(pl.col("mwh_total").median().alias("mwh_ref"))
)

df = (
    df.join(referencia_ceros, on=["cod_postal", "hora", "dia_semana", "mes"], how="left")
    .with_columns(
        pl.when(pl.col("mwh_total") == 0)
          .then(pl.col("mwh_ref"))
          .otherwise(pl.col("mwh_total"))
          .alias("mwh_total")
    )
    .drop("mwh_ref")
)

print(f"Ceros restantes: {df.filter(pl.col('mwh_total') == 0).shape[0]}")
print(f"Nulls mwh_total: {df['mwh_total'].null_count()}")

> **Resumen del tratamiento de nulos:**

> - **Reasignación AN a X2**, los registros de la estación AN se reasignaron a X2 y se rellenaron con datos reales de Meteocat para temp y humedad (2019–2023).

> - **Sectores de consumo**, mwh_industria, mwh_residencial y mwh_servicios nulos imputados a 0 (sector no activo ese bloque). mwh_total recalculado como suma de los 4 sectores.

> - **Continuidad temporal**, 220 bloques añadidos: 52 del CP 08011 (7–20 ago 2025, fallo de reporte específico) y 168 del 19-ago-2025 global (42 CPs × 4 bloques). Imputados con mediana histórica por CP, hora, día semana, mes (2019–2024).

> - **mwh_total = 0**, 378 registros en 3 fechas de 2025 (26-jun, 30-ago, 07-sep). Fallo en OpenData, bloques 00:00, 06:00 y 12:00 en cero mientras el 18:00 era normal. Imputados con mediana histórica por CP, hora, día semana + mes (2019–2024).

> - **Nulos meteorológicos restantes**, viento e irradiancia (X2 sin sensores en todo el período), temp/humedad 2024–2025 (X2 inactiva).  conocidos, se mantienen para feature engineering.

Limitación conocida: los 378 registros con mwh_total = 0 imputados con mediana histórica presentan inconsistencia con sus sectores componentes (mwh_industria, mwh_residencial,
mwh_servicios permanecen en 0). Al representar el 0.09% del dataset y no usarse los sectores como features del modelo, el impacto en el forecasting es despreciable. Se documenta como deuda técnica para versiones futuras.

---
# <font color='#1B3A5C'>  **Análisis Descriptivo Variable Objetivo: mwh_total** </font>

In [ ]:
df.select('mwh_total').describe()

- Media (101,467 MWh) > mediana (90,608 MWh), diferencia de aprox 11k, sesgo positivo confirmado, la cola derecha de outliers extremos jala la media hacia arriba.
- IQR: 57,120 a 132,708 MWh, rango de consumo "normal" entre barrios y bloques horarios.
- Mínimo 130 MWh, registros de madrugada en barrios pequeños, valor plausible.
- Máximo 1,486,114 MWh, outlier extremo a revisar
- Std (59,563) representa, alta dispersión esperada dado que mezcla 42 códigos postales con perfiles muy distintos (industrial, residencial, turístico).

In [ ]:
vals_target = df['mwh_total'].drop_nulls().to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(vals_target, bins=80, color='#2A9D8F', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribución de mwh_total', fontweight='bold')
axes[0].set_xlabel('MWh')
axes[0].set_ylabel('Frecuencia')

axes[1].boxplot(vals_target, vert=False, patch_artist=True,
                boxprops=dict(facecolor='#2A9D8F', alpha=0.7),
                medianprops=dict(color='navy', linewidth=2),
                flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#E9C46A'))
axes[1].set_title('Boxplot de mwh_total', fontweight='bold')
axes[1].set_xlabel('MWh')

plt.tight_layout()
plt.show()

In [ ]:
print(f"sum(is.na(mwh_total)): {df['mwh_total'].null_count()}")

> **Observaciones:**
> La distribución de mwh_total presenta sesgo positivo con cola derecha pronunciada. 

- La mediana está muy por debajo de la media, indicando que la mayoría de bloques tienen consumo moderado pero existen picos elevados de alta magnitud. 

- Estos outliers identificados son de 1.48 M lo cual muestra uqe estos no pueden ser datos reales.

- La desviación estándar de 59 k muestra la alta dispersión , los valores están muy dispersos.

> No se sigue una distribución normal así que se usará Spearman para correlaciones y Kruskal-Wallis / Mann-Whitney para tests de hipótesis.



#### <font color='#C0392B'><b>Outliers</b></font>

Detección con umbral de 5× la mediana por CP. Se usa la mediana porque no se infla por los propios outliers que intenta detectar.

In [ ]:
# Detección: umbral 5× mediana + contexto de casos
umbrales = df.group_by("cod_postal").agg(pl.col("mwh_total").median().alias("mediana"))

print(df.join(umbrales, on="cod_postal")
    .filter(pl.col("mwh_total") > pl.col("mediana") * 5)
    .group_by("cod_postal")
    .agg(pl.len().alias("n_registros"))
    .sort("n_registros", descending=True))


> - 6 codigos superan el umbral de 5 veces su mediana. 08037 destaca con 450 registros muy por encima del resto.

> - Revisar caso a caso

In [ ]:

# Contexto de los 3 casos erróneos del umbral mediana × 5
casos = {
    "08030 — Sostenido (may 2023)": df.filter(
        (pl.col("cod_postal") == "08030") &
        (pl.col("datetime") >= pl.datetime(2023, 5, 5)) &
        (pl.col("datetime") < pl.datetime(2023, 5, 12))
    ).select(["datetime", "mwh_total", "mwh_industria"]).sort("datetime"),
    "08037 — Puntual (nov/dic 2024, feb 2025)": df.filter(
        (pl.col("cod_postal") == "08037") &
        ((pl.col("datetime") == pl.datetime(2024, 11, 20, 0)) |
         (pl.col("datetime") == pl.datetime(2024, 12, 22, 0)) |
         (pl.col("datetime") == pl.datetime(2025, 2, 24, 0)))
    ).select(["datetime", "mwh_total", "mwh_servicios"]),
    "08022 — Parcial (may 2023)": df.filter(
        (pl.col("cod_postal") == "08022") &
        (pl.col("datetime") >= pl.datetime(2023, 5, 29)) &
        (pl.col("datetime") < pl.datetime(2023, 5, 31))
    ).select(["datetime", "mwh_total", "mwh_residencial"]).sort("datetime"),
}
for nombre, resultado in casos.items():
    print(f"\n{'='*60}\n  {nombre}\n{'='*60}")
    print(resultado)



In [ ]:
# Casos adicionales detectados con umbral 400k
print("\n" + "="*60 + "\n  Valores > 400k\n" + "="*60)
print(df.filter(pl.col("mwh_total") > 400000)
    .select(["datetime", "cod_postal", "nombre_postal", "mwh_total"])
    .sort("mwh_total", descending=True).head(10))

for cp in ["08013", "08036", "08009", "08006", "08011", "08030"]:
    fecha_max = df.filter(pl.col("cod_postal") == cp).sort("mwh_total", descending=True).head(1)["datetime"][0]
    print(f"\n{'='*50}\nCP {cp}\n{'='*50}")
    print(df.filter(
        (pl.col("cod_postal") == cp) &
        (pl.col("datetime") >= fecha_max - pl.duration(days=3)) &
        (pl.col("datetime") <= fecha_max + pl.duration(days=3))
    ).select(["datetime", "mwh_total"]).sort("datetime"))

Errores de reporte, se imputan con mediana histórica 2022-2024:

- **08030 (5–11 may 2023, 28 bloques):** mwh_industria de 6 a 12 veces el habitual aproximadamente 50,000 durante 7 días consecutivos. Caída brusca al día siguiente sin gradualidad es un fallo de acumulación en reporte.
- **08037 (nov-2024/dic-2024/feb-2025, 3 bloques):** mwh_servicios de 1.46M vs aprox 35k habitual. Patrón idéntico en los 3 casos en el mismo bloque horario (00:00h), posible error de acumulación.
- **08022 (29–30 may 2023, 6 bloques):** bloques 00:00h normales pero 06:00, 12:00 y 18:00h inflados  de 10 a 15 veces. Posible error de distribución del consumo diario entre bloques.
- **08030 (9 ene 2025, 4 bloques):** salto abrupto de ~300k a 558-595k en un solo día sin contexto climático que lo justifique. Vuelta a normal al día siguiente.
- **08037 (jul–nov 2025, 475 bloques):** todos los sectores 6 a 7 veces el histórico entre 80-92k MWh durante 5 meses consecutivos. Imposible que sea demanda real.
- **08009 (2 jun 2025, 4 bloques):** todos los sectores inflados proporcionalmente ( aproximadamente 7 veces habitual) en un único día. Sin contexto.

Picos reales, se mantienen:

- **08013 (jul 2025):** crecimiento gradual durante varios días (aprox 563k máx). Patrón consistente con ola de calor de inicio de verano.
- **08036 (ago 2023):** valores sostenidos 3-4 días (aprox 535k máx). Gradualidad clara de subida y bajada, consistente con ola de calor.
- **08006 (jul 2025):** pico de 4 < 5 días con valores crecientes y decrecientes. Mismo patrón de ola de calor que 08013.
- **08011 (ene 2020):** valores altos sostenidos durante más de una semana.

#### <font color='#C0392B'><b>Imputación de Outliers</b></font>

Funciones de imputación por mediana histórica (2022-2024)

In [ ]:
def imputar_sector(df, cp, sector, filtro, keys=["hora", "mes"]):
    med = (
        df.filter((pl.col("cod_postal") == cp) & ~filtro)
        .group_by(keys)
        .agg(pl.col(sector).median().alias("_med"))
    )
    df = (df.join(med, on=keys, how="left")
        .with_columns(
            pl.when((pl.col("cod_postal") == cp) & filtro)
              .then(pl.col("_med")).otherwise(pl.col(sector)).alias(sector)
        ).drop("_med"))
    return df.with_columns(
        pl.when((pl.col("cod_postal") == cp) & filtro)
        .then(
            pl.col("mwh_industria") + pl.col("mwh_residencial") +
            pl.col("mwh_servicios") + pl.col("mwh_no_especificado").fill_null(0)
        )
        .otherwise(pl.col("mwh_total"))
        .alias("mwh_total")
    )

def imputar_todos_sectores(df, cp, cond, anios_ref, mes_ref, keys=["hora"]):
    mes_filter = pl.col("mes").is_in(mes_ref) if isinstance(mes_ref, list) else pl.col("mes") == mes_ref
    med = (
        df.filter((pl.col("cod_postal") == cp) & pl.col("anio").is_in(anios_ref) & mes_filter)
        .group_by(keys)
        .agg([pl.col("mwh_industria").median().alias("_ind"),
              pl.col("mwh_residencial").median().alias("_res"),
              pl.col("mwh_servicios").median().alias("_ser")])
    )
    df = (df.join(med, on=keys, how="left")
        .with_columns([
            pl.when(cond).then(pl.col("_ind")).otherwise(pl.col("mwh_industria")).alias("mwh_industria"),
            pl.when(cond).then(pl.col("_res")).otherwise(pl.col("mwh_residencial")).alias("mwh_residencial"),
            pl.when(cond).then(pl.col("_ser")).otherwise(pl.col("mwh_servicios")).alias("mwh_servicios"),
        ]).drop(["_ind", "_res", "_ser"]))
    return df.with_columns(
        pl.when(cond)
        .then(
            pl.col("mwh_industria") + pl.col("mwh_residencial") +
            pl.col("mwh_servicios") + pl.col("mwh_no_especificado").fill_null(0)
        )
        .otherwise(pl.col("mwh_total"))
        .alias("mwh_total")
    )

# 08030 — mwh_industria, 5-11 may 2023
df = imputar_sector(df, "08030", "mwh_industria",
    (pl.col("datetime") >= pl.datetime(2023, 5, 5)) & (pl.col("datetime") < pl.datetime(2023, 5, 12)))

# 08037 — mwh_servicios, 3 bloques puntuales
df = imputar_sector(df, "08037", "mwh_servicios",
    (pl.col("datetime") == pl.datetime(2024, 11, 20, 0)) |
    (pl.col("datetime") == pl.datetime(2024, 12, 22, 0)) |
    (pl.col("datetime") == pl.datetime(2025, 2, 24, 0)))

# 08022 — mwh_residencial, 29-30 may 2023
df = imputar_sector(df, "08022", "mwh_residencial",
    (pl.col("datetime") >= pl.datetime(2023, 5, 29)) & (pl.col("datetime") < pl.datetime(2023, 5, 31)))

# 08030 — todos sectores, 9 ene 2025
df = imputar_todos_sectores(df, "08030",
    cond=(pl.col("cod_postal") == "08030") & (pl.col("datetime").dt.date() == pl.date(2025, 1, 9)),
    anios_ref=[2022, 2023, 2024], mes_ref=1)

# 08037 — todos sectores, jul-nov 2025
df = imputar_todos_sectores(df, "08037",
    cond=(pl.col("cod_postal") == "08037") & (pl.col("anio") == 2025) &
         (pl.col("mes").is_in([7, 8, 9, 10, 11])) & (pl.col("mwh_total") > 100000),
    anios_ref=[2022, 2023, 2024], mes_ref=[7, 8, 9, 10, 11], keys=["hora", "mes", "dia_semana"])

# Fix mwh_no_especificado anómalo en 08037 + recálculo mwh_total
_cond_noesp = (
    (pl.col("cod_postal") == "08037") & (pl.col("anio") == 2025) &
    (pl.col("mes").is_in([7, 8, 9, 10, 11])) & (pl.col("mwh_no_especificado").is_not_null())
)
df = df.with_columns(
    pl.when(_cond_noesp)
    .then(pl.lit(None).cast(pl.Int64))
    .otherwise(pl.col("mwh_no_especificado"))
    .alias("mwh_no_especificado")
)
# Recalcular mwh_total tras anular mwh_no_especificado
df = df.with_columns(
    pl.when(_cond_noesp)
    .then(pl.col("mwh_industria") + pl.col("mwh_residencial") + pl.col("mwh_servicios"))
    .otherwise(pl.col("mwh_total"))
    .alias("mwh_total")
)

# 08009 — todos sectores, 2 jun 2025
df = imputar_todos_sectores(df, "08009",
    cond=(pl.col("cod_postal") == "08009") & (pl.col("datetime").dt.date() == pl.date(2025, 6, 2)),
    anios_ref=[2022, 2023, 2024], mes_ref=6)

print("Todos los outliers imputados")

In [ ]:
# Seguridad: recalcular mwh_total solo si quedó null por algún edge case
df = df.with_columns(
    pl.when(pl.col("mwh_total").is_null())
    .then(
        pl.col("mwh_industria").fill_null(0) + pl.col("mwh_residencial").fill_null(0) +
        pl.col("mwh_servicios").fill_null(0) + pl.col("mwh_no_especificado").fill_null(0)
    )
    .otherwise(pl.col("mwh_total"))
    .alias("mwh_total")
)

print(f"Valores > 400k restantes: {df.filter(pl.col('mwh_total') > 400000).shape[0]}")
print(df.filter(pl.col("mwh_total") > 400000)
    .select(["datetime", "cod_postal", "nombre_postal", "mwh_total", "mwh_industria", "mwh_residencial", "mwh_servicios"])
    .sort("mwh_total", descending=True).head(20))

vals_target = df['mwh_total'].drop_nulls().to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(vals_target, bins=80, color='#1B3A5C', edgecolor='white', linewidth=0.3, alpha=0.85)
axes[0].set_title('Distribución de mwh_total (tras imputación)', fontweight='bold')
axes[0].set_xlabel('MWh'); axes[0].set_ylabel('Frecuencia')
axes[1].boxplot(vals_target, vert=False, patch_artist=True,
                boxprops=dict(facecolor='#1B3A5C', alpha=0.7),
                medianprops=dict(color='#E76F51', linewidth=2),
                flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#1B3A5C'),
                whiskerprops=dict(color='#1B3A5C'), capprops=dict(color='#1B3A5C'))
axes[1].set_title('Boxplot de mwh_total (tras tratamiento outliers)', fontweight='bold')
axes[1].set_xlabel('MWh')
plt.tight_layout(); plt.show()

> - Tras imputar los 520 registros erróneos con mediana histórica (2022-2024), la distribución muestra una forma log-normal coherente con el comportamiento esperado del consumo eléctrico urbano. La mayoría de bloques se concentran entre 20k–150k MWh, con una cola derecha moderada que refleja los picos reales de demanda estival. Los valores máximos legítimos (olas de calor 08013, 08036, 08006; ola de frío 08011) se mantienen como parte de la variabilidad natural del sistema.



### <font color='#C0392B'><b>Descomposición de consumo energético</b></font>


In [ ]:
# Serie temporal global — suma de todos los CPs
serie_global = (
    df.group_by("datetime")
    .agg([
        pl.col("mwh_industria").sum(),
        pl.col("mwh_residencial").sum(),
        pl.col("mwh_servicios").sum(),
    ])
    .sort("datetime")
)

fechas = serie_global["datetime"].to_numpy()
ind    = serie_global["mwh_industria"].to_numpy()
res    = serie_global["mwh_residencial"].to_numpy()
ser    = serie_global["mwh_servicios"].to_numpy()

fig, ax = plt.subplots(figsize=(24, 5))
ax.stackplot(fechas, ind, res, ser,
             labels=["Industria", "Residencial", "Servicios"],
             colors=["#264653", "#2A9D8F", "#E9C46A"],
             alpha=0.8)
ax.set_title("Consumo por sector — Barcelona (todos los CPs)", fontweight="bold")
ax.set_ylabel("MWh")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
(
    df.group_by("cod_postal")
    .agg([
        pl.col("mwh_industria").sum(),
        pl.col("mwh_residencial").sum(),
        pl.col("mwh_servicios").sum(),
        pl.col("mwh_no_especificado").sum(),
    ])
    .with_columns([
        (pl.col("mwh_industria") / (pl.col("mwh_industria") + pl.col("mwh_residencial") + pl.col("mwh_servicios")) * 100).round(1).alias("pct_industria"),
        (pl.col("mwh_residencial") / (pl.col("mwh_industria") + pl.col("mwh_residencial") + pl.col("mwh_servicios")) * 100).round(1).alias("pct_residencial"),
        (pl.col("mwh_servicios") / (pl.col("mwh_industria") + pl.col("mwh_residencial") + pl.col("mwh_servicios")) * 100).round(1).alias("pct_servicios"),
    ])
    .select(["cod_postal", "pct_industria", "pct_residencial", "pct_servicios"])
    .sort("pct_industria", descending=True)
)

mwh_total es una variable compuesta, es la suma de consumo industrial, residencial, de servicios y no especificado. 

>Ningún código postal tiene un perfil puro: 

- servicios domina en 38 de 42 CPs, siendo el sector principal en toda Barcelona. 

- La industria solo es relevante en 3 o 4 CPs (08039, 08004, 08038 principalmente)

- Residencial alcanza su máximo en zonas periféricas como 08032 (70% residencial). Esta composición explica por qué los patrones temporales difieren entre barrios, dependiendo del código postal será más probable en tener picos en diferentes horarios.

#### <font color='#C0392B'><b>Análisis de Serie Temporal de Consumo</b></font>


In [ ]:
# Serie temporal global — suma de todos los CPs

serie_global = (
    df.group_by("datetime")
    .agg([
        pl.col("mwh_industria").sum(),
        pl.col("mwh_residencial").sum(),
        pl.col("mwh_servicios").sum(),
    ])
    .sort("datetime")
)

fechas = serie_global["datetime"].to_numpy()
ind    = serie_global["mwh_industria"].to_numpy()
res    = serie_global["mwh_residencial"].to_numpy()
ser    = serie_global["mwh_servicios"].to_numpy()

fig, ax = plt.subplots(figsize=(24, 5))
ax.stackplot(fechas, ind, res, ser,
             labels=["Industria", "Residencial", "Servicios"],
             colors=["#264653", "#2A9D8F", "#E9C46A"],
             alpha=0.8)
ax.set_title("Consumo por sector — Barcelona (todos los CPs)", fontweight="bold")
ax.set_ylabel("MWh")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

> - El consumo eléctrico de Barcelona muestra una estacionalidad anual clara con picos en verano (julio–agosto) y en invierno (diciembre–enero), consistente con el efecto combinado de aire acondicionado y calefacción. Los servicios dominan el consumo en todos los períodos, seguidos del residencial. La industria tiene un peso estable.

> - Se observa una caída notable en 2020 (COVID 19) y una recuperación progresiva hasta 2022. A partir de 2023 el consumo se estabiliza con niveles ligeramente superiores al período pre-pandemia.

In [ ]:
(df.group_by("cod_postal")
 .agg([
     pl.col("nombre_postal").drop_nulls().first().alias("nombre_postal"),
     pl.len().alias("n_registros"),
     pl.col("mwh_total").null_count().alias("nulls"),
     pl.col("mwh_total").mean().alias("mwh_mean"),
     pl.col("mwh_industria").sum().alias("sum_industria"),
     pl.col("mwh_residencial").sum().alias("sum_residencial"),
     pl.col("mwh_servicios").sum().alias("sum_servicios"),
     pl.col("mwh_total").sum().alias("sum_total"),
 ])
 .with_columns([
     ((pl.col("sum_industria") / pl.col("sum_total") * 100).round(1).cast(pl.Utf8) + pl.lit("%")).alias("pct_industria"),
     ((pl.col("sum_residencial") / pl.col("sum_total") * 100).round(1).cast(pl.Utf8) + pl.lit("%")).alias("pct_residencial"),
     ((pl.col("sum_servicios") / pl.col("sum_total") * 100).round(1).cast(pl.Utf8) + pl.lit("%")).alias("pct_servicios"),
 ])
 .drop(["sum_industria", "sum_residencial", "sum_servicios", "sum_total"])
 .sort("mwh_mean", descending=True)
)

In [ ]:
medianas = df.group_by("cod_postal").agg(pl.col("mwh_total").median().alias("mediana_cp"))

df.filter(pl.col("mwh_total") > 400000)\
  .group_by(["cod_postal", "nombre_postal"])\
  .agg([
      pl.len().alias("n_registros"),
      pl.col("mwh_total").max().alias("max_mwh"),
  ])\
  .join(medianas, on="cod_postal")\
  .sort("n_registros", descending=True)

Se seleccionan 4 códigos postales representativos de perfiles de consumo distintos, con cobertura completa (10.104 registros) y mínimos nulos:

- **08038 Montjuïc / Zona Franca**  por perfil industrial (36.7% industria), mayor proporción del dataset
- **08005 El Poblenou** por perfil mixto equilibrado, referente en literatura smart city Barcelona, 4 nulos
- **08032 El Carmel / El Guinardó**  perfil residencial dominante (70.3% residencial)
- **08002 Barri Gòtic** por perfil servicios/turístico (75.6% servicios), consumo estacional marcado

In [ ]:
CPS = {
    "08038": "Montjuïc / Zona Franca (Industrial)",
    "08005": "El Poblenou (Mixto)",
    "08032": "El Carmel / El Guinardó (Residencial)",
    "08002": "Barri Gòtic (Servicios/Turístico)",
}

In [ ]:
for cp, nombre in CPS.items():
    serie_cp = (
        df.filter(pl.col("cod_postal") == cp)
        .sort("datetime")
        .select(["datetime", "mwh_industria", "mwh_residencial", "mwh_servicios"])
    )

    fechas = serie_cp["datetime"].to_numpy()
    ind    = serie_cp["mwh_industria"].to_numpy()
    res    = serie_cp["mwh_residencial"].to_numpy()
    ser    = serie_cp["mwh_servicios"].to_numpy()

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.stackplot(fechas, ind, res, ser,
                 labels=["Industria", "Residencial", "Servicios"],
                 colors=["#264653", "#2A9D8F", "#E9C46A"],
                 alpha=0.8)
    ax.set_title(f"Consumo por sector — CP {cp} ({nombre})", fontweight="bold")
    ax.set_ylabel("MWh")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

> **Observaciones iniciales — Consumo por sector**
 
> - 08038 Zona Franca: perfil industrial visible y sostenido. Pico 2020 probablemente asociado a actividad que no se detuvieron durante el confinamiento.
> - 08005 El Poblenou servicios domina sobre industria y residencial, reflejo de la reconversión del distrito hacia tecnología y oficinas.
> - 08032 El Carmel balance residencial-servicios con industria casi nula. Aparente estacionalidad de verano muy marcada.
> - 08002 Barri Gòtic servicios aplastante. El desplome de 2020–2021 es el más pronunciado de los 4 CPs, señal directa del impacto del COVID en el turismo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=False)
fig.suptitle("Estacionalidad anual — consumo medio mensual por CP", fontweight="bold")

meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
         "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

for ax, (cp, nombre) in zip(axes.flatten(), CPS.items()):
    mensual = (
        df.filter(
            (pl.col("cod_postal") == cp) &
            (pl.col("anio") < 2026)
        )
        .group_by("mes")
        .agg(pl.col("mwh_total").mean().alias("mwh_medio"))
        .sort("mes")
    )

    ax.bar(mensual["mes"].to_numpy(), mensual["mwh_medio"].to_numpy(),
           color="#2A9D8F", alpha=0.85)
    ax.set_title(f"{cp} — {nombre}", fontsize=9, fontweight="bold")
    ax.set_ylabel("MWh medio")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(meses, fontsize=8)

plt.tight_layout()
plt.show()

> Los 4 CPs muestran patrones estacionales distintos según su perfil de consumo.
- Industrial (08038) y turístico (08002) pican en julio-agosto con 195k y 155k MWh respectivamente con caídas en marzo-abril, patrón consistente con mayor actividad de temporada y demanda de climatización.
- El residencial (08032) invierte el patrón: picos en enero y diciembre 65k MWh y mínimos en abril-mayo (48k MWh), sugiriendo mayor dependencia de calefacción que de refrigeración.
- El mixto (08005) se mantiene más estable a lo largo del año con un pico moderado en julio-agosto (150k MWh) y un segundo repunte en enero por consumo invernal.
- En magnitud absoluta, Zona Franca lidera con diferencia mientras El Carmel se mantiene en el rango más bajo al carecer de actividad industrial o turística significativa.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=False)
fig.suptitle("Patrón semanal — consumo medio por día y hora", fontweight="bold")

dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

for ax, (cp, nombre) in zip(axes.flatten(), CPS.items()):
    semanal = (
        df.filter(pl.col("cod_postal") == cp)
        .group_by(["dia_semana", "hora"])
        .agg(pl.col("mwh_total").mean().alias("mwh_medio"))
        .sort(["dia_semana", "hora"])
    )

    for hora_val, color in zip([0, 6, 12, 18], ["#264653", "#2A9D8F", "#E9C46A", "#E76F51"]):
        subset = semanal.filter(pl.col("hora") == hora_val)
        ax.plot(subset["dia_semana"].to_numpy(),
                subset["mwh_medio"].to_numpy(),
                marker="o", label=f"{hora_val}h", color=color)

    ax.set_title(f"{cp} — {nombre}", fontsize=9, fontweight="bold")
    ax.set_ylabel("MWh medio")
    ax.set_xticks(range(7))
    ax.set_xticklabels(dias, fontsize=8)
    ax.legend(fontsize=7)
    ax.axvspan(4.5, 6.5, alpha=0.08, color="gray", label="Fin de semana")

plt.tight_layout()
plt.show()

> El patrón semanal revela diferencias claras entre perfiles. 

- INDUSTRIAL - La Zona Franca (08038) registra la caída más pronunciada en fin de semana, con descensos de 60k MWh en los bloques 6h y 12h del domingo respecto a los días laborables, señal directa de paralización de actividad industrial. 
- SERVICIOS El Barri Gòtic (08002) también cae en domingo pero mantiene el bloque 18h elevado, coherente con actividad de sevicios y ocio nocturno que no para el fin de semana.
- RESIDENCIAL - SERVICIOS El Carmel (08032) es el más estable a lo largo de la semana: la diferencia entre martes y domingo es mínima (unosk 5 MWh), confirmando que el consumo residencial no responde al calendario laboral. 
- INDUSTRIA- RESIDENCIAL El Poblenou (08005) presenta un comportamiento intermedio con caída ligera en fin de semana.
- En cuanto a magnitudes, Zona Franca lidera con bloques de 12h que superan los 200k MWh en días laborables, seguida de Barri Gòtic (165k MWh) y Poblenou(158k MWh). El Carmel se mantiene muy por debajo (69k MWh en 18h), reflejo de su perfil mayoritariamente residencial.
- En todos los CPs el bloque 12h concentra el mayor consumo y el 0h el mínimo, confirmando el patrón intradiario como una de las señales más predictivas del dataset.

> Conlsuciones de decomposición de consumo:

- Los patrones temporales confirman que el consumo eléctrico en Barcelona responde a tres ciclos superpuestos: 
- Estos ciclos son consistentes y repetibles, lo que valida la viabilidad del forecasting: si el consumo fuera aleatorio, predecir sería imposible. La señal estructural existe y es suficientemente fuerte.
- La diferencia de comportamiento entre perfiles (industrial, residencial, turístico) confirma que cod_postal y la composición sectorial son features clave para el modelo ya que sin ellas, el modelo confundiría patrones de barrios con dinámicas completamente distintas.
- Las variables de calendario (hora, dia_semana, mes, es_finde, es_festivo) posiblemente sean features de alta importancia esperada para el modelado


#### <font color='#C0392B'><b>Descomposición STL</b></font>


In [ ]:
serie_global_stl = (
    df.group_by("datetime")
    .agg(pl.col("mwh_total").sum().alias("mwh_total"))
    .sort("datetime")
)

fechas     = serie_global_stl["datetime"].to_numpy()
serie_vals = serie_global_stl["mwh_total"].to_numpy()

res = STL(serie_vals, period=28, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
fig.suptitle("Descomposición STL — Barcelona global (todos los CPs)", fontweight="bold")

for ax, (nombre_comp, vals) in zip(axes, {
    "Serie original": serie_vals,
    "Tendencia":      res.trend,
    "Estacionalidad": res.seasonal,
    "Residuo":        res.resid,
}.items()):
    ax.plot(fechas, vals, color="#264653", linewidth=0.6)
    ax.set_ylabel(nombre_comp, fontsize=9)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


 - El componente de tendencia no muestra una direccion sostenida de largo plazo. Oscila
entre 3M y 6M MWh con ciclos anuales visibles pero sin crecimiento ni decrecimiento
sistematico, lo que indica que la serie es estacionaria en media.

- El componente estacional captura el ciclo semanal (period=28 bloques) con amplitud
estable a lo largo de todo el periodo, oscilando entre -2M y +2M MWh. Existe una amplitud
constante, independiente del nivel de la serie.

El residuo se mantiene cercano a 0 con picos puntuales que pueden ser eventos no
recurrentes (olas de calor, festivos, etc...)

> La serie parece tener patrón Tipo 2, estacionaria en media con estacionalidad
fuerte y regular, lo que valida el uso de SARIMA como baseline y la inclusion de variables de calendario como features principales del modelo.

#### Decomposición STL Barrio Gotic Residencial- SERVICIOS

In [ ]:
cp = "08002"
serie_cp = (
    df.filter(pl.col("cod_postal") == cp)
    .sort("datetime")
    .select(["datetime", "mwh_total"])
)

fechas     = serie_cp["datetime"].to_numpy()
serie_vals = serie_cp["mwh_total"].to_numpy()

In [ ]:
print(
    df.filter(
        (pl.col("cod_postal") == "08002") &
        (pl.col("datetime") >= pl.datetime(2025, 7, 1)) &
        (pl.col("datetime") < pl.datetime(2025, 11, 1))
    )
    .select(["datetime", "mwh_total", "mwh_industria", "mwh_residencial", "mwh_servicios"])
    .sort("mwh_total")
    .head(20)
)

In [ ]:
res = STL(serie_vals, period=28, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
fig.suptitle(f"Descomposición STL — CP {cp} ({CPS[cp]})", fontweight="bold")

for ax, (nombre_comp, vals) in zip(axes, {
    "Serie original": serie_vals,
    "Tendencia":      res.trend,
    "Estacionalidad": res.seasonal,
    "Residuo":        res.resid,
}.items()):
    ax.plot(fechas, vals, color="#264653", linewidth=0.6)
    ax.set_ylabel(nombre_comp, fontsize=9)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> - La descomposición STL del Barri Gòtic (08002) revela una serie con estructura predecible pero con picos externos importantes. La tendencia muestra el impacto  pronunciado del COVID de los 4 CPs analizados: caída desde 200k hasta 100k MWh, entre enero y abril 2020, reflejo  del colapso del turismo y la actividad comercial. La recuperación fue gradual y no se completó hasta 2023.

- La estacionalidad capturada corresponde al ciclo semanal (period=28). El ciclo anual queda absorbido en la tendencia como ondulaciones lentas recurrentes. En el Barri Gòtic, al ser un perfil de servicios/turístico, el pico anual se concentra en verano y no tiene un repunte invernal significativo. Este comportamiento contrasta con lo esperado en perfiles residenciales, donde la tendencia debería mostrar picos en enero-diciembre por calefacción.

- La estacionalidad semanal es clara y consistente, con amplitud variable ya que es más fuerte en verano oscilando 50k MWh cuando la actividad turística intensifica los contrastes entre bloques horarios. 

- El residuo se mantiene cercano a 0 la mayor parte del tiempo, con picos puntuales en 2022-2024 atribuibles a eventos no recurrentes (Tal vez conciertos o festivales)

#### Decomposición STL resto de barrios

In [ ]:
for cp, nombre in {k: v for k, v in CPS.items() if k != "08002"}.items():
    serie_cp = (
        df.filter(pl.col("cod_postal") == cp)
        .sort("datetime")
        .select(["datetime", "mwh_total"])
    )

    fechas     = serie_cp["datetime"].to_numpy()
    serie_vals = serie_cp["mwh_total"].to_numpy()

    res = STL(serie_vals, period=28, robust=True).fit()

    fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
    fig.suptitle(f"Descomposición STL — CP {cp} ({nombre})", fontweight="bold")

    for ax, (nombre_comp, vals) in zip(axes, {
        "Serie original": serie_vals,
        "Tendencia":      res.trend,
        "Estacionalidad": res.seasonal,
        "Residuo":        res.resid,
    }.items()):
        ax.plot(fechas, vals, color="#264653", linewidth=0.6)
        ax.set_ylabel(nombre_comp, fontsize=9)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

> Barcelona global (todos los CPs):

- La tendencia no muestra direccion sostenida de largo plazo: oscila entre 3M y 6M MWh
  con ciclos anuales pero sin crecimiento ni decrecimiento sistematico.
- La amplitud estacional es constante a lo largo de todo el periodo, lo que confirma
  que el modelo aditivo es apropiado.
- El residuo se mantiene cercano a 0 salvo picos puntuales atribuibles a eventos no recurrentes.

> 08002 Barri Gotic (Servicios/Turistico):

- Tendencia con la caida COVID mas pronunciada de los 4 CPs: de 200k baja a 80k MWh,
  reflejo del colapso del turismo y la actividad comercial.
- La estacionalidad se reduce durante el confinamiento y se recupera gradualmente hasta 2022.
- Residuo limpio salvo picos puntuales en 2022-2024.

> 08038 Zona Franca (Industrial):

- Tendencia inestable con pico anomalo en nov 2019 y caida COVID moderada.
- La amplitud de la estacionalidad crece notablemente a partir de 2024-2025, senal de
  mayor variabilidad en la actividad industrial reciente.
- Residuo el mas ruidoso de los 4 CPs, con picos frecuentes en 2022-2025.

> 08005 El Poblenou (Mixto):

- Tendencia sin direccion clara, caida COVID moderada con recuperacion lenta.
- Estacionalidad semanal estable y consistente a lo largo de toda la serie.
- Residuo el mas limpio de los 4 CPs, confirmando que es el perfil mas predecible.

> 08032 El Carmel (Residencial):

- Tendencia con ciclos anuales invertidos respecto al resto: picos en invierno y valles
  en primavera, coherente con dependencia de calefaccion mas que de refrigeracion.
- Impacto COVID el mas suave de los 4, ya que el consumo residencial no depende de
  actividad economica externa.
- Estacionalidad de menor amplitud y mas estable que los perfiles industriales o turisticos.
- Residuo muy cercano a 0, anticipando menores errores de prediccion en el modelado.

### <font color='#C0392B'><b>ADF Y KPSS</b></font>


- ADF: Augmented Dickey-Fuller. Detecta si la serie tiene raíz unitaria (tendencia que no vuelve a un valor fijo). H0: la serie NO es estacionaria.

- KPSS: Kwiatkowski-Phillips-Schmidt-Shin. Lo contrario: H0: la serie SÍ es estacionaria.

Se usan juntos porque se contradicen, si ambos coinciden la conclusión es clara, si no coinciden hay estacionariedad parcial.

In [ ]:
from arch.unitroot import ADF, KPSS

CPS_ADF = {
    "08038": "Montjuïc / Zona Franca (Industrial)",
    "08005": "El Poblenou (Mixto)",
    "08032": "El Carmel / El Guinardó (Residencial)",
    "08002": "Barri Gòtic (Servicios/Turístico)",
}

resultados = []

# Serie global
serie_global = (
    df.group_by("datetime")
    .agg(pl.col("mwh_total").sum())
    .sort("datetime")["mwh_total"]
    .to_numpy()
)

adf_g  = ADF(serie_global, method="aic")
kpss_g = KPSS(serie_global)

resultados.append({
    "CP": "Global",
    "Barrio": "Barcelona (todos los CPs)",
    "ADF stat": round(adf_g.stat, 3),
    "ADF p-valor": round(adf_g.pvalue, 4),
    "ADF estacionaria": "Si" if adf_g.pvalue < 0.05 else "No",
    "KPSS stat": round(kpss_g.stat, 3),
    "KPSS p-valor": round(kpss_g.pvalue, 4),
    "KPSS estacionaria": "Si" if kpss_g.pvalue > 0.05 else "No",
})

# 4 CPs representativos
for cp, nombre in CPS_ADF.items():
    serie = (
        df.filter(pl.col("cod_postal") == cp)
        .sort("datetime")["mwh_total"]
        .to_numpy()
    )

    adf   = ADF(serie, method="aic")
    kpss_ = KPSS(serie)

    resultados.append({
        "CP": cp,
        "Barrio": nombre,
        "ADF stat": round(adf.stat, 3),
        "ADF p-valor": round(adf.pvalue, 4),
        "ADF estacionaria": "Si" if adf.pvalue < 0.05 else "No",
        "KPSS stat": round(kpss_.stat, 3),
        "KPSS p-valor": round(kpss_.pvalue, 4),
        "KPSS estacionaria": "Si" if kpss_.pvalue > 0.05 else "No",
    })

pd.DataFrame(resultados)

> **La serie es parcialmente estacionaria: estacionaria en media pero no en varianza.**

> Estacionariedad en media (ADF)

- ADF rechaza la hipotesis nula en los 5 casos (p < 0.05).
- Las series son estacionarias en media. A pesar de eventos como COVID o la estacionalidad anual, el consumo siempre orbita alrededor de un nivel central estable sin una tendencia sostenida.

> Estacionariedad en varianza (KPSS)

- KPSS rechaza la hipotesis nula en los 5 casos (p < 0.05).
- La varianza no es constante. Los cambios del fenómeno visibles en el STL (caida COVID 2020, recuperacion gradual, mayor variabilidad industrial en 2024-2025) explican por que el tamano de los saltos cambia entre periodos.

> Conclusion

- Ambos resultados son coherentes entre si. Las series estacionarias en media con cambios estructurales en varianza.
- Para SARIMA esto implica que se requerira diferenciacion estacional pero no diferenciacion regular.
- XGBoost y LSTM/GRU son robustos a esta condicion por naturaleza y no requieren
  transformacion previa.

#### <font color='#C0392B'><b>Análisis ACF y PACF</b></font>

In [ ]:
serie = (
    df.group_by("datetime")
    .agg(pl.col("mwh_total").sum())
    .sort("datetime")["mwh_total"]
    .to_numpy()
)

def acf_manual(x, nlags=112):
    n = len(x)
    x = x - x.mean()
    acf_vals = [np.dot(x[:n-k], x[k:]) / np.dot(x, x) for k in range(nlags+1)]
    return np.array(acf_vals)

def pacf_manual(x, nlags=112):
    acf_vals = acf_manual(x, nlags)
    pacf_vals = [1.0]
    for k in range(1, nlags+1):
        mat = np.array([[acf_vals[abs(i-j)] for j in range(k)] for i in range(k)])
        vec = acf_vals[1:k+1]
        try:
            coef = np.linalg.solve(mat, vec)
            pacf_vals.append(coef[-1])
        except:
            pacf_vals.append(0)
    return np.array(pacf_vals)

lags      = np.arange(113)
acf_vals  = acf_manual(serie, nlags=112)
pacf_vals = pacf_manual(serie, nlags=112)
ci        = 1.96 / np.sqrt(len(serie))

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].vlines(lags, 0, acf_vals, color="#1B3A5C", linewidth=0.8)
axes[0].axhline(0,   color="black", linewidth=0.5)
axes[0].axhline(ci,  color="gray",  linestyle="--", linewidth=0.8)
axes[0].axhline(-ci, color="gray",  linestyle="--", linewidth=0.8)
axes[0].axvline(x=4,  color="#E76F51", linestyle="--", alpha=0.7, label="lag 4 = 24h")
axes[0].axvline(x=28, color="#2A9D8F", linestyle="--", alpha=0.7, label="lag 28 = 7 dias")
axes[0].set_title("ACF — Barcelona Global", fontweight="bold")
axes[0].set_xlabel("Lag (bloques de 6h)")
axes[0].legend()

axes[1].vlines(lags, 0, pacf_vals, color="#1B3A5C", linewidth=0.8)
axes[1].axhline(0,   color="black", linewidth=0.5)
axes[1].axhline(ci,  color="gray",  linestyle="--", linewidth=0.8)
axes[1].axhline(-ci, color="gray",  linestyle="--", linewidth=0.8)
axes[1].axvline(x=4,  color="#E76F51", linestyle="--", alpha=0.7, label="lag 4 = 24h")
axes[1].axvline(x=28, color="#2A9D8F", linestyle="--", alpha=0.7, label="lag 28 = 7 dias")
axes[1].set_title("PACF — Barcelona Global", fontweight="bold")
axes[1].set_xlabel("Lag (bloques de 6h)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print(f"{'lag':>5} │ {'ACF':>8} │ {'PACF':>8}")
print("──────┼──────────┼──────────")
for i in range(30):
    print(f"  {i:3d} │ {acf_vals[i]:>8.4f} │ {pacf_vals[i]:>8.4f}")


> ACF (Autocorrelacion)

- Mide qué tan parecido es el consumo de hoy al de hace X bloques, acumulando
  todo el efecto cadena entre lags intermedios.
- Lag 4 (24h, 0.83): el consumo de hace 24h es muy parecido al actual. El mismo
  bloque horario del dia anterior explica gran parte de la variabilidad.
- Lag 28 (7 dias, 0.87): el mismo bloque de hace 7 dias es el mas parecido de
  toda la serie. El patron semanal domina sobre el diario.
- Lags 8, 12, 16, 20, 24 (entre 0.70 y 0.78): picos repetidos cada 4 lags
  confirman que el ciclo diario de 24h persiste a lo largo de toda la semana.
- Lag 2 (12h, -0.05): correlacion ligeramente negativa. El bloque de hace 12h
  suele ser el opuesto al actual (si ahora es mediodia, hace 12h era madrugada).
- Lags 56 y 84: ecos del ciclo semanal (mismo bloque hace 2 y 3 semanas).
  La similitud se mantiene significativa pero decrece progresivamente.

> PACF (Autocorrelacion parcial)

- Mide cuanto explica el consumo de hace X bloques sobre el de hoy, una vez
  descontado lo que ya explican todos los lags anteriores. Es la contribucion
  directa e independiente de cada lag.
- Lag 1 (6h, 0.37): contribucion directa moderada del bloque inmediatamente anterior.
- Lag 2 (12h, -0.22): contribucion directa negativa, el bloque de hace 12h tiene
  efecto inverso propio, consistente con el contraste entre madrugada y mediodia.
- Lag 3 (18h, 0.47): contribucion directa relevante, el bloque de hace 18h aporta
  informacion propia mas alla de los lags anteriores.
- Lag 4 (24h, 0.75): el mas alto de todos. El mismo bloque del dia anterior tiene
  la contribucion directa mas fuerte, independiente del efecto cadena.
- Lag 5 (30h, -0.65): correccion negativa fuerte tras el pico de lag 4, el modelo
  ajusta hacia abajo para no sobreestimar el efecto del dia anterior.
- Lag 29 (7 dias + 6h, -0.40): ajuste del ciclo semanal justo al completar la semana.
- A partir de lag 6 hasta lag 28 las contribuciones son moderadas o ruido.


> Conclusion

- Los lags mas informativos para el modelo son lag_1 (6h antes), lag_4 (24h antes,mismo bloque del dia anterior) y lag_28 (7 dias antes, mismo bloque de la semana anterior).
- Los lags 8 (2 dias), 12 (3 dias), 16 (4 dias) de la ACF son en gran parte efecto acumulado del ciclo diario, no contribuciones directas independientes, por eso no aparecen destacados en PACF.
- Lag_3 (18h antes) y lag_5 (30h antes) aportan contribucion directa relevante que se considerara en la configuracion de SARIMA (parametros AR y MA).

> Implicaciones para feature engineering

- Entran como features:
  -  lag_1  (6h antes): contribucion directa confirmada por PACF (0.37). El bloque
    inmediatamente anterior siempre aporta senal propia.
  -  lag_4  (24h antes): el mas importante de todos, PACF 0.75. Contribucion directa
    mas fuerte de la serie.
  -  lag_28  (7 dias antes): ACF 0.87, el mas parecido de toda la serie. Captura el
    patron semanal completo.
  -  rolling_mean_7d : promedio de los ultimos 7 dias. Suaviza la senal y captura la
    tendencia reciente sin depender de un solo bloque puntual.

- No entran como features:
  -  lag_8, lag_12, lag_16  (2, 3, 4 dias): PACF los descarta. Lo que parecen aportar
    ya esta explicado por lag_4 y lag_1 encadenados. Meterlos seria redundancia pura.
  -  lag_3, lag_5  (18h y 30h): tienen contribucion directa en PACF pero son artefactos
    del ciclo diario. Para SARIMA si son relevantes como parametros AR, pero para
    XGBoost y LSTM no aportan nada que lag_1 y lag_4 juntos no capturen ya.
  -  lag_56, lag_84  (2 y 3 semanas): la correlacion existe pero decrece progresivamente
    y lag_28 ya captura el patron semanal. Coste computacional sin ganancia real.

---
# <font color='#1B3A5C'>  **Análisis Descriptivo de variables Numéricas** </font>

In [ ]:
VARS_NUM = ['mwh_industria', 'mwh_residencial', 'mwh_servicios', 'mwh_no_especificado',
            'lst_celsius', 'temp_mean', 'temp_max', 'temp_min',
            'humedad_mean', 'viento_mean', 'precipitacion_sum', 'irradiancia_mean']

df_num = df.select(VARS_NUM)
df_num.head(6)

In [ ]:
df_num.describe()

Sectores de consumo
- mwh_industria, std (19k) mayor que la media (8.7k) y max (434k) es 50× la mediana (1.7k). Refleja la heterogeneidad entre barrios: Zona Franca vs barrios sin actividad industrial.
- mwh_residencial y mwh_servicios son más simétricas ya que la media y la mediana son parecidas. Servicios domina en magnitud media (56k).
- mwh_no_especificado,  85.35% nulos, count real de solo 62k de 424k registros.

Temperatura
- temp_mean rango -0.26°C a 36°C, media 17.8°C, mediana 17.4°C, distribución casi simétrica, sin anomalías.
- temp_max y temp_min pueden ser colineales con temp_mean, candidatas a descartar en feature engineering.
- lst_celsius media 24.7°C vs temp_mean 17.9°C, diferencia de aproximadamente 7°C esperada por Urban Heat Island (asfalto y tejados siempre más calientes que el aire). No son redundantes porque capturan señales distintas.

Variables problemáticas
- precipitacion_sum, mediana y P75 = 0.0, más del 75% de bloques sin lluvia. Más útil como binaria (llueve/no_llueve).

- irradiancia_mean, mínimo de -4.27 W/m², error de sensor ay qeu esto es físicamente imposible. se debe topar a 0 en feature engineering.

- humedad_mean y viento_mean, sin anomalías destacables.

In [ ]:
df.select(VARS_NUM + ['mwh_total']).null_count()

- mwh_industria, mwh_residencial, mwh_servicios, mwh_total, 0 nulos, imputación completada en limpieza.
- mwh_no_especificado, 362,256 nulos (85.35%), esto es normal y esperado
- temp_mean, humedad_mean, 28,025 nulos (6.6%), corresponden a X2 inactiva en períodos puntuales.
- temp_max, temp_min, precipitacion_sum, 136,700 nulos (32.2%), estaciones sin cobertura completa.
- viento_mean, irradiancia_mean,, 153,400 nulos (36.1%), X2 sin sensores para estas variables en todo el período.
- lst_celsius, 169,044 nulos (39.8%), cobertura de nubes (MODIS satelital), algo esperado
- Todos los nulos restantes son estructurales y conocidos, se mantienen para feature engineering.

- Nulos estructurales conocidos:
  - Los candidatos a imputación son temp_mean y humedad_mean(28,025 nulos, 6.6%), ya que tienen cobertura histórica 2019-2023 completa y solo fallan en X2 durante 2024-2025. 
  - El resto (viento, irradiancia, precipitacion, lst_celsius) se evaluarán durante el análisis bivariante para decidir si su variación entre barrios puede justificar imputarlos o se gestionan directamente en feature engineering.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

VARS_NUM_PLOT = ['mwh_industria', 'mwh_residencial', 'mwh_servicios',
                 'lst_celsius', 'temp_mean', 'temp_max', 'temp_min',
                 'humedad_mean', 'viento_mean', 'precipitacion_sum', 'irradiancia_mean']

n_cols = 3
n_rows = int(np.ceil(len(VARS_NUM_PLOT) / n_cols))

fig = plt.figure(figsize=(16, n_rows * 5))
fig.suptitle('Distribución de variables numéricas', fontsize=13, fontweight='bold', y=1.02)

for i, var in enumerate(VARS_NUM_PLOT):
    v = df[var].drop_nulls().to_numpy()
    
    ax1 = fig.add_subplot(n_rows * 2, n_cols, (i // n_cols) * n_cols * 2 + (i % n_cols) + 1)
    ax1.hist(v, bins=30, color='#264653', edgecolor='white', alpha=0.85)
    ax1.set_title(var, fontsize=9, fontweight='bold')
    ax1.set_ylabel('Frecuencia', fontsize=7)
    ax1.tick_params(axis='x', rotation=45, labelsize=7)
    
    ax2 = fig.add_subplot(n_rows * 2, n_cols, (i // n_cols) * n_cols * 2 + n_cols + (i % n_cols) + 1)
    ax2.boxplot(v, vert=False, patch_artist=True,
                boxprops=dict(facecolor='#264653', alpha=0.7),
                medianprops=dict(color='#E9C46A', linewidth=2),
                flierprops=dict(marker='o', markersize=1.5, alpha=0.3))
    ax2.tick_params(axis='x', rotation=45, labelsize=7)  # ← aquíierprops=dict(marker='o', markersize=1.5, alpha=0.3))

plt.tight_layout()
plt.show()

- mwh_industria, mwh_residencial y mwh_servicios, todos con sesgo positivo, producto de la
  heterogeneidad entre barrios y picos energéticos puntuales que jalan la cola derecha.
- lst_celsius, temp_mean, temp_max, temp_min, distribuciones aparentemente bimodales,
  dos picos visibles que pueden explicarse por diferencias entre barrios o por estacionalidad (invierno vs verano), o ambos efectos superpuestos. Se revisará en el análisis bivariante por CP y por mes.

- humedad_mean, la más cercana a normal de todas las variables, con un leve sesgo negativo hacia valores altos de humedad.

- viento_mean, sesgo positivo claro, la mayoría de bloques con viento bajo y cola derecha de eventos puntuales de viento fuerte.

- precipitacion_sum, casi todo en 0 por clima mediterráneo, BCN simplemente no llueve la mayoría de días. El boxplot revela que cuando sí llueve los valores son dispersos y variables.

- irradiancia_mean, gran masa en 0 por bloques nocturnos (00h y 06h), ceros reales y esperados, no imputados. El resto distribuido ampliamente hasta 917 W/m². Comportamiento bimodal día/noche.

Test de Normalidad

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for i, var in enumerate(VARS_NUM_PLOT):
    v = df[var].drop_nulls().to_numpy()
    stats.probplot(v, dist='norm', plot=axes[i])
    axes[i].get_lines()[0].set(color='#264653', alpha=0.4, markersize=2)
    axes[i].get_lines()[1].set(color='#E76F51', linewidth=2)
    axes[i].set_title(var, fontsize=9, fontweight='bold')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('QQ-plots — variables numéricas', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Ninguna variable sigue distribución normal, confirmado por la desviación sistemática
de la línea de referencia en todos los QQ-plots.

- mwh_industria, mwh_residencial, mwh_servicios, curvas en S pronunciadas con colas
  derechas muy pesadas, sesgo positivo extremo ya visto en los histogramas.

- lst_celsius, temp_mean, temp_max, temp_min, son las más cercanas a la línea pero
  muestran un ligero "baile" en forma de S suave, consistente con la bimodalidad
  observada en los histogramas (mezcla de estaciones del año y diferencias entre barrios).

- humedad_mean, la más cercana a normal visualmente, pero con leve desviación
  en las colas confirmando el sesgo izquierdo ya identificado.

- viento_mean, cola derecha clara, los eventos extremos de viento se alejan
  significativamente de la línea.

- precipitacion_sum, escalón casi vertical cerca de 0, reflejo directo de la
  masa de ceros. No tiene distribución continua real.

- irradiancia_mean, escalón pronunciado en 0 por los bloques nocturnos,
  seguido de distribución dispersa. La forma en S invertida confirma la bimodalidad día/noche.

Conclusión: se descarta Pearson para correlaciones, se usará **Spearman**. Para tests de hipótesis, **Kruskal-Wallis y Mann-Whitney** en lugar de ANOVA.

#### <font color='#C0392B'><b>Análisis Bivariante</b></font>

Los sectores mwh_industria, mwh_residencial y mwh_servicios se excluyen del análisis bivariante por ser componentes directos de mwh_total, su separación Q1/Q3 sería trivial. Se añaden temp_max y temp_min para completar el análisis meteorológico.

In [ ]:
VARS_BIN = ['temp_mean', 'lst_celsius', 'humedad_mean',
            'irradiancia_mean', 'viento_mean', 'precipitacion_sum']

ETIQUETAS = {
    'temp_mean':         ('Temperatura media del aire (°C)',    C1),
    'lst_celsius':       ('LST — Temperatura superficial (°C)', C2),
    'humedad_mean':      ('Humedad relativa (%)',                C4),
    'irradiancia_mean':  ('Irradiancia solar media (W/m²)',     C5),
    'viento_mean':       ('Velocidad media del viento (m/s)',    C3),
    'precipitacion_sum': ('Precipitacion acumulada (mm)',        C6),
}

UMBRAL = {
    'temp_mean': 50, 'lst_celsius': 50, 'humedad_mean': 50,
    'irradiancia_mean': 50, 'viento_mean': 50, 'precipitacion_sum': 30,
}

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()
fig.suptitle('Relacion entre variables meteorologicas y consumo electrico',
             fontweight='bold', fontsize=13, y=1.01)

for i, var in enumerate(VARS_BIN):
    ax     = axes[i]
    xlabel, color = ETIQUETAS[var]

    datos = (
        df.select([var, 'mwh_total'])
        .drop_nulls()
        .to_pandas()
    )

    datos['bin'] = pd.cut(datos[var], bins=25)
    resumen = (
        datos.groupby('bin', observed=True)['mwh_total']
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    resumen['centro'] = resumen['bin'].apply(lambda x: x.mid)
    resumen['se']     = resumen['std'] / resumen['count'].pow(0.5)
    resumen['ci']     = 1.96 * resumen['se']
    resumen = resumen[resumen['count'] > UMBRAL[var]]

    norm = plt.Normalize(resumen['mean'].min(), resumen['mean'].max())
    cmap = plt.cm.YlOrRd

    for _, row in resumen.iterrows():
        ax.bar(row['centro'],
               row['mean'],
               width=(resumen['centro'].iloc[1] - resumen['centro'].iloc[0]) * 0.85,
               color=cmap(norm(row['mean'])),
               alpha=0.25,
               zorder=1)

    ax.plot(resumen['centro'], resumen['mean'],
            color=color, linewidth=2.5,
            marker='o', markersize=4,
            zorder=3)

    ax.scatter(resumen['centro'], resumen['mean'],
               c=resumen['mean'], cmap='YlOrRd',
               vmin=resumen['mean'].min(),
               vmax=resumen['mean'].max(),
               s=40, zorder=4, edgecolors='white', linewidths=0.5)

    n_ticks   = 8
    idx_ticks = np.linspace(0, len(resumen) - 1, n_ticks, dtype=int)
    ax.set_xticks(resumen['centro'].iloc[idx_ticks])
    ax.set_xticklabels(
        [f"{resumen['centro'].iloc[j]:.1f}" for j in idx_ticks],
        fontsize=8, rotation=0
    )

    media_global = datos['mwh_total'].mean()
    ax.axhline(media_global, color='gray', linewidth=0.8,
               linestyle='--', alpha=0.6)
    ax.text(resumen['centro'].iloc[-1], media_global * 1.005,
            'media global', fontsize=7, color='gray', ha='right')

    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel('mwh_total medio', fontsize=8)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k')
    )

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()
fig.suptitle('Distribucion de consumo por rango de variable meteorologica',
             fontweight='bold', fontsize=13)

for i, var in enumerate(VARS_BIN):
    ax     = axes[i]
    xlabel = ETIQUETAS[var][0]

    datos = (
        df.select([var, 'mwh_total'])
        .drop_nulls()
        .to_pandas()
    )

    datos['bin'] = pd.cut(datos[var], bins=10)
    agrupado     = datos.groupby('bin', observed=True)
    grupos       = [g['mwh_total'].values for _, g in agrupado]
    medianas     = [np.median(g) for g in grupos]
    labels       = [str(round(b.mid, 1)) for b in agrupado.groups.keys()]

    norm = plt.Normalize(min(medianas), max(medianas))
    cmap = plt.cm.YlOrRd

    bp = ax.boxplot(
        grupos, labels=labels, patch_artist=True,
        medianprops=dict(color='white', linewidth=2),
        flierprops=dict(marker='o', markersize=2, alpha=0.3,
                        markerfacecolor='gray', markeredgewidth=0),
        whiskerprops=dict(linewidth=1.2, linestyle='--', color='gray'),
        capprops=dict(linewidth=1.5, color='gray'),
        boxprops=dict(linewidth=0.5),
        widths=0.6,
    )

    for box, med in zip(bp['boxes'], medianas):
        box.set_facecolor(cmap(norm(med)))
        box.set_alpha(0.85)

    for j, med in enumerate(medianas):
        ax.scatter(j + 1, med, color='white', s=25,
                   zorder=5, edgecolors=cmap(norm(med)), linewidths=1.5)

    ax.plot(range(1, len(medianas) + 1), medianas,
            color='gray', linewidth=1, linestyle='--', alpha=0.5, zorder=3)

    media_global = datos['mwh_total'].mean()
    ax.axhline(media_global, color='gray', linewidth=0.8, linestyle=':', alpha=0.6)
    ax.text(len(grupos) + 0.1, media_global * 1.01,
            'media\nglobal', fontsize=6, color='gray', va='bottom')

    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel('mwh_total', fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax.tick_params(axis='x', labelsize=7, rotation=45)

plt.tight_layout()
plt.show()

> Relacion consumo variables meteorologicas (binned mean)

- temp_mean y lst_celsius muestran relacion no lineal con el consumo. Patron en J
  para temp_mean (minimo en aprox 15°C, subida fuerte a partir de 25°C) y en U para lst_celsius (minimo en aprox 20°C, subida en ambos extremos).
- humedad_mean presenta relacion inversa moderada. Los dias muy secos (aprox 15%) coinciden con olas de calor y registran consumo mas alto.
- irradiancia_mean es practicamente plana entre 0 y 600 W/m², con caida en valores extremos atribuible a pocos registros en ese rango.
- viento_mean y precipitacion_sum muestran relacion debil o nula con el consumo medio, confirmando su utilidad limitada como variables continuas.

> Dispersion por rango (boxplot)

- La dispersion dentro de cada rango es muy alta en todas las variables: los bigotes alcanzan 400-500k MWh independientemente del valor de la variable meteorologica.
- Esto confirma que las variables meteorologicas son predictores relevantes pero no suficientes por si solas.
- La mediana sube progresivamente con la temperatura en temp_mean y lst_celsius,
  visible en el gradiente de color de las cajas de amarillo a rojo oscuro.

In [ ]:
from scipy.ndimage import uniform_filter1d

VARS_TREND = ['temp_mean', 'lst_celsius', 'humedad_mean', 'irradiancia_mean']

COLORES_ANIO = {
    2019: '#264653', 2020: '#2A9D8F', 2021: '#8AB17D',
    2022: '#E9C46A', 2023: '#F4A261', 2024: '#E76F51', 2025: '#C1121F',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
fig.suptitle('Evolucion anual de variables meteorologicas (2019-2025)',
             fontweight='bold', fontsize=13)

for i, var in enumerate(VARS_TREND):
    ax = axes[i]

    anual = (
        df.filter(pl.col(var).is_not_null() & (pl.col('anio') < 2026))
        .group_by('anio')
        .agg([
            pl.col(var).median().alias('mediana'),
            pl.col(var).quantile(0.25).alias('q1'),
            pl.col(var).quantile(0.75).alias('q3'),
        ])
        .sort('anio')
        .to_pandas()
    )

    anios   = anual['anio'].values
    medianas = anual['mediana'].values
    colores  = [COLORES_ANIO[a] for a in anios]

    # Banda IQR suave
    ax.fill_between(anios, anual['q1'], anual['q3'],
                    color=C1, alpha=0.08)

    # Linea de conexion neutra
    ax.plot(anios, medianas, color='gray',
            linewidth=1, linestyle='--', alpha=0.4, zorder=1)

    # Puntos coloreados por año con tamaño destacado
    for x, y, c in zip(anios, medianas, colores):
        ax.scatter(x, y, color=c, s=120,
                   zorder=4, edgecolors='white', linewidths=1.5)

    # Etiqueta de valor encima de cada punto
    for x, y, c in zip(anios, medianas, colores):
        ax.text(x, y + (anual['q3'].max() - anual['q1'].min()) * 0.03,
                f'{y:.1f}', ha='center', fontsize=8,
                color=c, fontweight='bold')
        

    # Linea de tendencia lineal
    from numpy.polynomial import polynomial as P
    coef    = np.polyfit(anios, medianas, 1)
    y_trend = np.polyval(coef, anios)
    
    color_trend = '#C1121F' if coef[0] > 0 else '#264653'
    
    ax.plot(anios, y_trend,
            color=color_trend, linewidth=1.5,
            linestyle='-', alpha=0.5, zorder=2)

    # Anotacion de pendiente
    pendiente = coef[0]
    signo     = '▲' if pendiente > 0 else '▼'
    ax.text(anios[-1], y_trend[-1],
            f'  {signo} {abs(pendiente):.2f}/año',
            fontsize=7, color=color_trend,
            va='center', alpha=0.8)

    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_ylabel('Mediana anual', fontsize=8)
    ax.set_xticks(anios)
    ax.set_xticklabels([str(a) for a in anios], fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}'))

    # Leyenda con colores de año
    handles = [plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=COLORES_ANIO[a], markersize=8,
               label=str(a)) for a in anios]
    ax.legend(handles=handles, fontsize=7, loc='upper right',
              ncol=2, framealpha=0.4)

plt.tight_layout()
plt.show()

> Evolucion anual de variables meteorologicas (2019-2025)

- temp_mean muestra tendencia ascendente de +0.31°C/año. Pasó de 16.4°C en 2019 a 18.3°C en 2025, acumulando casi 2°C en el periodo analizado. Consistente con el calentamiento
  urbano progresivo documentado en Barcelona.
- humedad_mean desciende -0.31%/año de 67.2% en 2019 a 66.3% en 2025. La relacion inversa temperatura-humedad es coherente fisicamente con el calentamiento del aire.
- lst_celsius de manera más ligera (+0.11°C/año), con mayor variabilidad interanual quetemp_mean. El pico de 2022 (27.9°C) contrasta con la caida en 2025 (25.6°C), explicada
  por el sesgo de cobertura MODIS en invierno documentado anteriormente.
- irradiancia_mean presenta tendencia ligeramente descendente (-0.22 W/m²/año) pero con alta variabilidad interanual.
- Con 7 puntos anuales las tendencias son indicativas y se documentan como contexto para la interpretacion del modelo.

In [ ]:
VARS_TREND  = ['temp_mean', 'lst_celsius', 'humedad_mean', 'irradiancia_mean']
MESES_LABEL = ['Ene','Feb','Mar','Abr','May','Jun',
                'Jul','Ago','Sep','Oct','Nov','Dic']

COLORES_ANIO = {
    2019: '#264653', 2020: '#2A9D8F', 2021: '#8AB17D',
    2022: '#E9C46A', 2023: '#F4A261', 2024: '#E76F51', 2025: '#C1121F',
}

ANIOS = sorted(df.filter(pl.col('anio') < 2026)['anio'].unique().to_list())

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()
fig.suptitle('Evolucion mensual de variables meteorologicas por año',
             fontweight='bold', fontsize=13)

for i, var in enumerate(VARS_TREND):
    ax = axes[i]

    m_2019 = (
        df.filter(pl.col(var).is_not_null() & (pl.col('anio') == 2019))
        .group_by('mes').agg(pl.col(var).median().alias('mediana'))
        .sort('mes').to_pandas()
    )
    m_2025 = (
        df.filter(pl.col(var).is_not_null() & (pl.col('anio') == 2025))
        .group_by('mes').agg(pl.col(var).median().alias('mediana'))
        .sort('mes').to_pandas()
    )

    # Banda solo en meses comunes
    meses_comunes = sorted(set(m_2019['mes']) & set(m_2025['mes']))
    m_2019_f = m_2019[m_2019['mes'].isin(meses_comunes)].sort_values('mes')
    m_2025_f = m_2025[m_2025['mes'].isin(meses_comunes)].sort_values('mes')

    y_min = np.minimum(m_2019_f['mediana'].values, m_2025_f['mediana'].values)
    y_max = np.maximum(m_2019_f['mediana'].values, m_2025_f['mediana'].values)
    sube  = m_2025_f['mediana'].values > m_2019_f['mediana'].values

    ax.fill_between(m_2019_f['mes'], y_min, y_max,
                    where=sube,
                    color='#C1121F', alpha=0.08, label='_nolegend_')
    ax.fill_between(m_2019_f['mes'], y_min, y_max,
                    where=~sube,
                    color='#264653', alpha=0.08, label='_nolegend_')

    for anio in ANIOS:
        mensual = (
            df.filter(pl.col(var).is_not_null() & (pl.col('anio') == anio))
            .group_by('mes')
            .agg(pl.col(var).median().alias('mediana'))
            .sort('mes')
            .to_pandas()
        )

        lw    = 2.5 if anio in [2019, 2025] else 1.2
        alpha = 1.0 if anio in [2019, 2025] else 0.45
        ms    = 5   if anio in [2019, 2025] else 3

        ax.plot(mensual['mes'], mensual['mediana'],
                color=COLORES_ANIO[anio],
                linewidth=lw, alpha=alpha,
                marker='o', markersize=ms,
                label=str(anio),
                zorder=5 if anio in [2019, 2025] else 2)

    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_ylabel('Mediana', fontsize=8)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MESES_LABEL, fontsize=8)

    handles = [plt.Line2D([0], [0], color=COLORES_ANIO[a],
               linewidth=2, label=str(a)) for a in ANIOS]
    ax.legend(handles=handles, fontsize=7,
              loc='upper right', ncol=2, framealpha=0.4)

plt.tight_layout()
plt.show()

> Evolucion mensual por año (2019-2025)

- temp_mean confirma la tendencia ascendente mes a mes.l La linea de 2025 (rojo) se situa consistentemente por encima de 2019 (verde oscuro) en los meses de primavera e inicio de verano, con la banda roja entre ambas años visible en mayo-junio.
- lst_celsius muestra el mismo patron en primavera pero con la banda azul en enero-marzo, donde 2025 esta por debajo de 2019. Esto es el sesgo de cobertura MODIS en invierno ya documentado, no un enfriamiento real.
- humedad_mean presenta mayor variabilidad interanual que las variables termicas, la banda roja (2025 > 2019) aparece en enero-febrero y la banda azul (2025 < 2019) en mayo-agosto, sin un patron estacional claro y consistente entre años.
- irradiancia_mean es la mas ruidosa de las cuatro. La linea de 2025 destaca con picos anomalos en mayo-junio (aprox 150 W/m²) que no se repiten en otros años, probablemente asociados a periodos excepcionales de cielo despejado o a artefactos de cobertura MODIS.
- El patron estacional de todas las variables se mantiene estable entre años (verano siempre mas calido, invierno mas frio), lo que confirma que la estacionalidad es robusta y predecible para el modelo.

> Temperatura y LST

- temp_mean y lst_celsius muestran distribucion bimodal en el histograma univariante,
  el KDE por estacion confirma que son dos poblaciones superpuestas: invierno (pico ~12°C)
  y verano (pico ~25°C).
- En el bivariante Q1 vs Q3, el consumo alto presenta ambos picos (frio y calor) mientras
  el consumo bajo se concentra en temperaturas intermedias, confirmando la relacion en U
  con mwh_total. Senal predictiva fuerte.
- lst_celsius aporta senal complementaria a temp_mean, no redundante: temp_mean mide
  temperatura del aire desde la estacion Meteocat (igual para todos los CPs de la misma
  estacion), mientras lst_celsius varia por CP capturando diferencias intraurbanas por
  Urban Heat Island. Ambas se mantienen en el modelo.

> Humedad

- Sin bimodal: el KDE muestra las 4 estaciones solapadas en torno a 60-75%.
- El bivariante Q1 vs Q3 revela que el consumo alto se desplaza hacia humedad baja,
  coherente con dias calurosos y secos de verano que generan mayor demanda de refrigeracion.

> Viento

- Las 4 estaciones son practicamente identicas en el KDE, sin patron estacional.
- Las distribuciones Q1 vs Q3 se solapan casi completamente.
- Variable con menor poder discriminante del grupo.

> Irradiancia

- El KDE expone la causa de su bimodal: el pico en 0 corresponde a invierno (dias cortos,
  bloques nocturnos), mientras la cola hasta 800 W/m² es verano.
- El bivariante Q1 vs Q3 queda distorsionado por la masa de ceros nocturnos; requiere
  analisis separado por bloque horario diurno.

> Precipitacion

- Todas las estaciones pegadas al 0 en el KDE, sin separacion en el bivariante.
- Confirma su utilidad limitada como variable continua; candidata a binarizar (llueve / no llueve).

> Limitacion MODIS LST

- Se utilizo MOD11A1 (satelite Terra, paso ~10:30h local), asignando el mismo valor LST
  a todos los bloques del dia.
- El producto MYD11A1 (satelite Aqua, ~13:30h local) capturaría mejor el pico de Urban
  Heat Island al coincidir con el maximo de radiacion solar.
- Ambos productos podrian combinarse asignando Terra al bloque 06h-12h y Aqua al bloque
  12h-18h, obteniendo LST especifica por bloque diurno. Se documenta como linea futura.

#### <font color='#C0392B'><b>Test de Hipótesis Mann Whitney U</b></font>

In [ ]:
# Mann-Whitney: variables numéricas vs mwh_total (Q1 vs Q3)
q1_t = df['mwh_total'].quantile(0.25)
q3_t = df['mwh_total'].quantile(0.75)

VARS_MW = ['temp_mean', 'temp_max', 'temp_min', 'humedad_mean',
           'viento_mean', 'precipitacion_sum', 'irradiancia_mean', 'lst_celsius']

resultados_mw = []
for var in VARS_MW:
    df_pair = df.select([var, 'mwh_total']).drop_nulls()
    g_bajo = df_pair.filter(pl.col('mwh_total') <= q1_t)[var].to_numpy()
    g_alto = df_pair.filter(pl.col('mwh_total') >= q3_t)[var].to_numpy()
    stat_mw, p_mw = stats.mannwhitneyu(g_bajo, g_alto, alternative='two-sided')
    resultados_mw.append({
        'variable':      var,
        'mediana_bajo':  round(float(np.median(g_bajo)), 3),
        'mediana_alto':  round(float(np.median(g_alto)), 3),
        'p_value':       p_mw,
        'significativo': 'si' if p_mw < 0.05 else 'no'
    })

res_mw = sorted(resultados_mw, key=lambda x: x['p_value'])

print(f"{'variable':<25} {'mediana_bajo':>14} {'mediana_alto':>14} {'p_value':>14} {'sig':>5}")
print('-' * 80)
for r in res_mw:
    print(f"{r['variable']:<25} {r['mediana_bajo']:>14.3f} {r['mediana_alto']:>14.3f} {r['p_value']:>14.4e} {r['significativo']:>5}")


> Todas las variables son significativas (p < 0.05) pero con diferencias practicas muy distintas entre ellas.
- irradiancia_mean muestra la mayor diferencia: 193 W/m² en consumo alto vs 7.3 W/m² en bajo, reflejo del patron horario diurno mas que una relacion causal directa.
- temp_mean y lst_celsius confirman la relacion termica: consumo alto aparece en dias mas calidos (19.0°C vs 16.6°C y 27.2°C vs 24.6°C respectivamente).
- humedad_mean presenta relacion inversa moderada: consumo alto en dias mas secos (65.4% vs 70.9%), coherente con olas de calor.
- viento_mean es significativa pero la diferencia practica es minima (2.1 vs 2.3 m/s), lo que cuestiona su utilidad como feature continua.
- precipitacion_sum tiene ambas medianas en 0. Lla informacion util esta en si llueve o no, no en la cantidad exacta.

---
# <font color='#1B3A5C'>  **Análisis Descriptivo de variables Categóricas** </font>

In [ ]:
# Describe variables categóricas/temporales
VARS_CAT = ['hora', 'dia_semana', 'mes', 'anio', 'es_finde', 'es_festivo']

for var in VARS_CAT:
    conteo = (
        df.group_by(var)
        .agg(
            pl.len().alias('n'),
            pl.col('mwh_total').mean().round(1).alias('mwh_medio')
        )
        .sort(var)
        .with_columns((pl.col('n') / len(df) * 100).round(2).alias('pct'))
    )
    print(f"\n{'='*50}")
    print(f"  {var}")
    print(f"{'='*50}")
    print(conteo)

# Describe cod_postal
print(f"\n{'='*50}")
print(f"  cod_postal")
print(f"{'='*50}")
print(
    df.filter(pl.col('nombre_postal').is_not_null())
    .group_by(['cod_postal', 'nombre_postal'])
    .agg([
        pl.len().alias('n'),
        pl.col('mwh_total').mean().round(1).alias('mwh_medio'),
        pl.col('mwh_total').median().round(1).alias('mwh_mediana'),
        pl.col('mwh_total').max().round(1).alias('mwh_max'),
        pl.col('mwh_industria').sum().alias('sum_ind'),
        pl.col('mwh_residencial').sum().alias('sum_res'),
        pl.col('mwh_servicios').sum().alias('sum_serv'),
    ])
    .with_columns([
        (pl.col('sum_ind') / (pl.col('sum_ind') + pl.col('sum_res') + pl.col('sum_serv')) * 100).round(1).alias('pct_ind'),
        (pl.col('sum_res') / (pl.col('sum_ind') + pl.col('sum_res') + pl.col('sum_serv')) * 100).round(1).alias('pct_res'),
        (pl.col('sum_serv') / (pl.col('sum_ind') + pl.col('sum_res') + pl.col('sum_serv')) * 100).round(1).alias('pct_serv'),
    ])
    .drop(['sum_ind', 'sum_res', 'sum_serv'])
    .sort('mwh_medio', descending=True)
)

> Patrones temporales

- Hora: valle en 00h (71k MWh) y pico en 12h (120k MWh), diferencia del 69%.
  Distribucion balanceada al 25% por bloque, confirma grilla temporal completa.
- Dia semana: caida progresiva hacia el fin de semana, laborables aprox 105-106k MWh
  vs sabado 91k y domingo 86k. Diferencia de 20k MWh entre miercoles y domingo.
- Mes: doble pico en enero (110k MWh) y julio (118k MWh), valle en abril (89k MWh).
  Confirma estacionalidad por calefaccion invernal y refrigeracion estival.
- Anio: caida notable en 2020 (99.7k MWh) vs 2019 (113.3k MWh) por COVID.
  Recuperacion parcial en 2021-2022 y tendencia descendente desde 2023.
- es_finde: fin de semana reduce el consumo un 11% (106k vs 94k MWh).
- es_festivo: festivos reducen un 15% (101k vs 86k MWh), mayor impacto que el fin
  de semana por paralizacion simultanea de industria, servicios y comercio.

> Codigo postal

- Rango de consumo extremo: desde 29.7k MWh en 08033 Vallbona hasta 233.6k MWh en
  08040 Zona Franca, diferencia de 8x entre minimo y maximo.
- Los CPs con mayor componente industrial lideran: Zona Franca (08040, 29% ind.),
  Montjuic (08038, 37% ind.) y El Bon Pastor (08030, 16% ind.).
- En el extremo opuesto, barrios residenciales perifericos: El Carmel (08032, 70% res.),
  Vilapicina (08031, 64% res.) y Vallbona (08033, 57% res.).


In [ ]:
VARS_BINARIAS = {
    'es_finde':   {'valores': [0, 1], 'labels': ['Laborable', 'Fin de semana'],
                   'colores': [C1, C4]},
    'es_festivo': {'valores': [0, 1], 'labels': ['No festivo', 'Festivo'],
                   'colores': [C1, C5]},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Consumo electrico — laborable vs fin de semana y festivos',
             fontweight='bold', fontsize=13)

for row, (var, cfg) in enumerate(VARS_BINARIAS.items()):
    grupos  = [df.filter(pl.col(var) == v)['mwh_total'].drop_nulls().to_numpy()
               for v in cfg['valores']]
    labels  = cfg['labels']
    colores = cfg['colores']
    medias  = [g.mean() for g in grupos]
    medianas = [np.median(g) for g in grupos]

    # Barplot con valor encima
    ax_bar = axes[row, 0]
    bars = ax_bar.bar(labels, medias, color=colores, alpha=0.85,
                      edgecolor='white', linewidth=0.5, width=0.5)
    for bar, val in zip(bars, medias):
        ax_bar.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() * 1.01,
                    f'{val/1000:.1f}k', ha='center',
                    fontsize=10, fontweight='bold',
                    color=bar.get_facecolor())
    dif = (medias[0] - medias[1]) / medias[0] * 100
    ax_bar.text(0.98, 0.95, f'Reduccion: {dif:.1f}%',
                transform=ax_bar.transAxes, ha='right', va='top',
                fontsize=9, color='gray')
    ax_bar.set_title(f'Consumo medio — {var}', fontsize=10, fontweight='bold')
    ax_bar.set_ylabel('MWh medio', fontsize=8)
    ax_bar.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax_bar.set_ylim(0, max(medias) * 1.15)

    # Boxplot
    ax_box = axes[row, 1]
    bp = ax_box.boxplot(grupos, labels=labels, patch_artist=True,
                        showfliers=False,
                        medianprops=dict(color='white', linewidth=2),
                        whiskerprops=dict(linewidth=1.2, linestyle='--', color='gray'),
                        capprops=dict(linewidth=1.5, color='gray'),
                        boxprops=dict(linewidth=0.5),
                        widths=0.4)
    for patch, color in zip(bp['boxes'], colores):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
    for j, med in enumerate(medianas):
        ax_box.scatter(j + 1, med, color='white', s=40,
                       zorder=5, edgecolors=colores[j], linewidths=1.5)

    media_global = df['mwh_total'].mean()
    ax_box.axhline(media_global, color='gray', linewidth=0.8,
                   linestyle=':', alpha=0.6)
    ax_box.text(2.3, media_global * 1.01, 'media global',
                fontsize=7, color='gray')
    ax_box.set_title(f'Distribucion — {var}', fontsize=10, fontweight='bold')
    ax_box.set_ylabel('MWh', fontsize=8)
    ax_box.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

> es_finde y es_festivo

- Fin de semana reduce el consumo aproximadamente un 11% respecto a laborables
  (94.2k vs 106.2k MWh).
- Festivos reducen aproximadamente un 15% (86.3k vs 101.7k MWh), mayor impacto
  que el fin de semana porque paralizan simultaneamente industria, servicios y
  comercio, mientras que el sabado mantiene parte de la actividad de servicios.
- El boxplot confirma que la reduccion no es solo en media sino en toda la
  distribucion: el IQR de fin de semana y festivos se desplaza hacia abajo,
  con medianas de aprox 90k y 86k MWh respectivamente.
- La dispersion es similar en todos los grupos, la variabilidad no cambia segun
  el tipo de dia, solo baja el nivel general de consumo.

In [ ]:
CPS_ESTACIONAL = {
    "08040": "Zona Franca",
    "08038": "Montjuïc",
    "08002": "Barri Gòtic",
    "08036": "Antiga Esquerra",
    "08032": "El Carmel",
    "08033": "Vallbona",
}

COLORES_ESTACIONAL = {
    "08040": C1,
    "08038": C2,
    "08002": C3,
    "08036": C4,
    "08032": C5,
    "08033": C6,
}

meses_label = ["Ene","Feb","Mar","Abr","May","Jun",
                "Jul","Ago","Sep","Oct","Nov","Dic"]

df_sel  = df.filter(pl.col("cod_postal").is_in(list(CPS_ESTACIONAL.keys())))
mensual = (
    df_sel
    .group_by(["cod_postal", "mes"])
    .agg(pl.col("mwh_total").mean().alias("mwh_medio"))
    .sort(["cod_postal", "mes"])
)

fig, ax = plt.subplots(figsize=(14, 6))

for cp, nombre in CPS_ESTACIONAL.items():
    serie = (
        mensual.filter(pl.col("cod_postal") == cp)
        .sort("mes")
    )
    meses = serie["mes"].to_list()
    vals  = serie["mwh_medio"].to_list()

    ax.plot(meses, vals,
            color=COLORES_ESTACIONAL[cp],
            marker='o', markersize=5,
            linewidth=2)

    ax.text(12.15, vals[-1], f'{cp} · {nombre}',
            fontsize=7.5, color=COLORES_ESTACIONAL[cp],
            va='center', fontweight='bold')

ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_label, fontsize=9)
ax.set_ylabel('MWh medio por bloque de 6h', fontsize=9)
ax.set_title('Consumo mensual medio por barrio (2019-2025)',
             fontweight='bold', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.set_xlim(1, 14.5)

plt.tight_layout()
plt.show()

> Consumo mensual medio por barrio (6 perfiles)

- Los barrios industriales (08040 Zona Franca, 08038 Montjuic) mantienen consumo
  alto y estable todo el año con leve pico invernal, la produccion continua amortigua la estacionalidad.
- Los barrios de servicios (08002 Barri Gotic, 08036 Antiga Esquerra) muestran el
  pico de julio mas pronunciado y caida marcada en abril-mayo, patron tipico de climatizacion estival.
- Los residenciales (08032 El Carmel, 08033 Vallbona) siguen la curva en U con picos en enero y julio pero con amplitud mucho menor por su bajo volumen absoluto.
- La separacion entre Zona Franca (aprox 270k MWh en julio) y Vallbona (aprox 30k MWh) reproduce la heterogeneidad de 8x entre barrios ya documentada, justificando que el modelo trate cada CP como serie temporal independiente.

### <font color='#C0392B'><b>Varianza de barrios</b></font>


In [ ]:
varianza_cp = (
    df.group_by("cod_postal")
    .agg([
        pl.col("mwh_total").std().alias("std"),
        pl.col("mwh_total").mean().alias("media"),
        pl.col("mwh_industria").sum().alias("sum_ind"),
        pl.col("mwh_residencial").sum().alias("sum_res"),
        pl.col("mwh_servicios").sum().alias("sum_serv"),
        pl.col("nombre_postal").first(),
    ])
    .with_columns([
        (pl.col("std") / pl.col("media") * 100).round(1).alias("cv_pct"),
        (pl.col("sum_ind") / (pl.col("sum_ind") + pl.col("sum_res") + pl.col("sum_serv")) * 100).round(1).alias("pct_ind"),
        (pl.col("sum_res") / (pl.col("sum_ind") + pl.col("sum_res") + pl.col("sum_serv")) * 100).round(1).alias("pct_res"),
        (pl.col("sum_serv") / (pl.col("sum_ind") + pl.col("sum_res") + pl.col("sum_serv")) * 100).round(1).alias("pct_serv"),
    ])
    .select(["cod_postal", "nombre_postal", "media", "std", "cv_pct", "pct_ind", "pct_res", "pct_serv"])
    .sort("cv_pct", descending=True)
)
print(varianza_cp)

In [ ]:
vcp_pd = varianza_cp.to_pandas()

fig, ax = plt.subplots(figsize=(12, 14))  # más alto

norm    = plt.Normalize(vcp_pd['cv_pct'].min(), vcp_pd['cv_pct'].max())
colores_cv = [C5 if cv > 50 else C3 if cv > 35 else C1
              for cv in vcp_pd['cv_pct']]

bars = ax.barh(vcp_pd['nombre_postal'], vcp_pd['cv_pct'],
               color=colores_cv, alpha=0.85,
               edgecolor='white', linewidth=0.3,
               height=0.7)  # barras más delgadas para evitar solapamiento

ax.axvline(35, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(50, color=C5,    linewidth=0.8, linestyle='--', alpha=0.5)
ax.text(35.5, 41, 'CV=35%', fontsize=7, color='gray')
ax.text(50.5, 41, 'CV=50%', fontsize=7, color=C5)

for bar, val in zip(bars, vcp_pd['cv_pct']):
    color_txt = C5 if val > 50 else C3 if val > 35 else C1
    ax.text(bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=7,
            color=color_txt, fontweight='bold')

ax.set_xlabel('Coeficiente de variacion (%)', fontsize=9)
ax.set_title('Variabilidad de consumo por barrio (CV = std/media)',
             fontweight='bold', fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.invert_yaxis()
ax.set_xlim(0, 80)

plt.tight_layout()
plt.show()

> Variabilidad de consumo por barrio (CV)

- Los barrios con mayor volatilidad son de servicios: Sant Antoni (CV=71.6%),
  L'Antiga Esquerra (CV=58.8%) y Dreta de l'Eixample (CV=54-58%). Su consumo
  se concentra en bloques diurnos laborables y colapsa en madrugadas y festivos,
  generando una dispersion intrinsecamente alta.
- Los barrios industriales y portuarios presentan los CV mas bajos: La Zona Franca
  (CV=23.2%), El Port (CV=23.2%) y La Verneda (CV=21.8%). La produccion continua
  en turnos suaviza los picos y valles del calendario.
- Esta heterogeneidad tiene implicaciones directas para el modelado: los barrios
  con CV alto concentraran los errores mas grandes y la tasa de deteccion de picos
  sera especialmente exigente en esos CPs.

### <font color='#C0392B'><b>Análisis Bivariante de Variables Categóricas</b></font>


In [ ]:
consumo_pd = (
    df.filter(pl.col('nombre_postal').is_not_null())
    .group_by(['cod_postal', 'nombre_postal'])
    .agg(pl.col('mwh_total').mean().round(1).alias('mwh_medio'))
    .sort('mwh_medio', descending=True)
    .to_pandas()
)

# Etiqueta con CP + nombre
consumo_pd['label'] = consumo_pd['cod_postal'] + ' · ' + consumo_pd['nombre_postal']

fig, ax = plt.subplots(figsize=(14, 10))

norm    = plt.Normalize(consumo_pd['mwh_medio'].min(),
                        consumo_pd['mwh_medio'].max())
colores = [plt.cm.YlOrRd(norm(v)) for v in consumo_pd['mwh_medio']]

bars = ax.barh(consumo_pd['label'], consumo_pd['mwh_medio'],
               color=colores, alpha=0.85,
               edgecolor='white', linewidth=0.3)

for bar, val in zip(bars, consumo_pd['mwh_medio']):
    ax.text(bar.get_width() + 500,
            bar.get_y() + bar.get_height() / 2,
            f'{val/1000:.1f}k', va='center', fontsize=7.5,
            fontweight='bold', color='#333333')

ax.set_xlabel('MWh medio por bloque de 6h', fontsize=9)
ax.set_title('Consumo medio por barrio (2019-2025)',
             fontweight='bold', fontsize=12)
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k')
)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

> Consumo medio por barrio (2019-2025)

- Rango de 8 veces entre el minimo y el maximo: desde 29.7k MWh en 08033 Vallbona
  hasta 233.6k MWh en 08040 La Zona Franca.
- Los tres CPs con mayor consumo son industriales o portuarios: Zona Franca (08040), El Bon Pastor (08030) y Zona Universitaria (08028).
- Los CPs con consumo bajo del centro (08010 Pl. Catalunya, 08008-08009 Dreta de
  l'Eixample) no reflejan baja actividad economica sino que son codigos postales de superficie reducida que cubren solo una fraccion del barrio.
- La heterogeneidad de 8x justifica tratar cod_postal como variable categorica:
  un modelo sin distincion espacial promediaria perfiles completamente distintos.

### <font color='#C0392B'><b>Heat Maps</b></font>


In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

# Agrupar por mes todos los CPs
todos_cp = (
    df.filter(pl.col('nombre_postal').is_not_null())
    .group_by(['cod_postal', 'mes'])
    .agg(pl.col('mwh_total').mean().alias('mwh_medio'))
    .sort(['cod_postal', 'mes'])
)

cps_validos = [
    cp for cp in df['cod_postal'].unique().to_list()
    if todos_cp.filter(pl.col('cod_postal') == cp).shape[0] == 12
]

# Matriz normalizada
matriz = np.zeros((len(cps_validos), 12))
for i, cp in enumerate(cps_validos):
    vals = todos_cp.filter(pl.col('cod_postal') == cp).sort('mes')['mwh_medio'].to_numpy()
    vmin, vmax = vals.min(), vals.max()
    matriz[i] = (vals - vmin) / (vmax - vmin) if vmax > vmin else vals

# Ordenar por clustering jerarquico — barrios similares juntos
linkage_matrix = linkage(pdist(matriz, metric='euclidean'), method='ward')
orden = dendrogram(linkage_matrix, no_plot=True)['leaves']
matriz_ord  = matriz[orden]
cps_ord     = [cps_validos[i] for i in orden]

nombres_cp = (
    df.filter(pl.col('nombre_postal').is_not_null())
    .select(['cod_postal', 'nombre_postal'])
    .unique()
    .to_pandas()
    .set_index('cod_postal')['nombre_postal']
    .to_dict()
)
labels_y    = [f"{cp} · {(nombres_cp.get(cp) or '')[:16]}" for cp in cps_ord]
meses_label = ['Ene','Feb','Mar','Abr','May','Jun',
                'Jul','Ago','Sep','Oct','Nov','Dic']

fig, ax = plt.subplots(figsize=(12, 14))

im = ax.imshow(matriz_ord, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

# Lineas de separacion entre celdas
for x in range(13):
    ax.axvline(x - 0.5, color='white', linewidth=0.8, alpha=0.6)
for y in range(len(cps_ord) + 1):
    ax.axhline(y - 0.5, color='white', linewidth=0.4, alpha=0.4)

ax.set_xticks(range(12))
ax.set_xticklabels(meses_label, fontsize=10, fontweight='bold')
ax.set_yticks(range(len(cps_ord)))
ax.set_yticklabels(labels_y, fontsize=7.5)
ax.set_title('Patron estacional por barrio, consumo normalizado (0=min, 1=max)',
             fontweight='bold', fontsize=12, pad=12)

cbar = plt.colorbar(im, ax=ax, label='Intensidad relativa', shrink=0.35)
cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

> Patron estacional por barrio (heatmap normalizado)

- No existe un unico patron estacional en Barcelona: el clustering jerarquico
  identifica al menos dos regimenes diferenciados.
- Los barrios de servicios (08036 L'Antiga Esquerra, 08002 Barri Gotic, 08013
  La Sagrada Familia) concentran su pico en julio, respondiendo a la refrigeracion estival.
- Los barrios con mayor componente industrial o portuaria (08040 Zona Franca, 08038 Montjuic, 08039 El Port) muestran pico en enero, dominados por calefaccion y turnos continuos de produccion.
- Un tercer grupo (08011 Sant Antoni, 08034 Pedralbes) presenta patrones fragmentados
  sin mes dominante claro, coherente con los coeficientes de variacion mas altos
  identificados en el analisis de variabilidad.
- Esta heterogeneidad refuerza la necesidad de tratar cod_postal como variable
  categorica: un modelo global promediaria regimenes opuestos y perderia la senal
  estacional de ambos grupos.

In [ ]:
pivot = (
    df.group_by(['dia_semana', 'hora'])
    .agg(pl.col('mwh_total').mean().alias('mwh_medio'))
    .sort(['dia_semana', 'hora'])
)

pivot_pd = pivot.to_pandas().pivot(index='dia_semana', columns='hora', values='mwh_medio')
pivot_pd.index   = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
pivot_pd.columns = ['00h', '06h', '12h', '18h']

fig, ax = plt.subplots(figsize=(9, 6))

im = ax.imshow(pivot_pd.values, aspect='auto',
               cmap='YlOrRd', vmin=pivot_pd.values.min() * 0.95)

# Sin rayas — solo separadores sutiles entre celdas
for x in np.arange(-0.5, 4, 1):
    ax.axvline(x, color='white', linewidth=1.5, alpha=0.6)
for y in np.arange(-0.5, 7, 1):
    ax.axhline(y, color='white', linewidth=1.5, alpha=0.6)

# Valores en negro o blanco segun fondo
umbral = pivot_pd.values.max() * 0.70
for i in range(7):
    for j in range(4):
        val   = pivot_pd.values[i, j]
        color = 'white' if val > umbral else '#333333'
        ax.text(j, i, f'{val/1000:.1f}k',
                ha='center', va='center',
                fontsize=10, fontweight='bold', color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('MWh medio', fontsize=9)
cbar.ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k')
)

ax.set_xticks(range(4))
ax.set_xticklabels(pivot_pd.columns, fontsize=10)
ax.set_yticks(range(7))
ax.set_yticklabels(pivot_pd.index, fontsize=10)
ax.set_title('Consumo por bloque horario y dia de semana',
             fontweight='bold', fontsize=12, pad=12)
ax.set_xlabel('Hora del bloque', fontsize=9)
ax.set_ylabel('Dia de la semana', fontsize=9)

plt.tight_layout()
plt.show()

> Consumo por bloque horario y dia de semana

- El bloque nocturno (00h) es practicamente constante toda la semana. Tiene rango de 69.1k a 71.9k MWh, diferencia inferior al 4%. La madrugada homogeniza patrones independientemente del tipo de dia.
- A medida que avanza el dia en dias qeu se trabaja vs fin de semana se amplifica. En el bloque de 12h la diferencia entre miercoles (128.3k MWh) y domingo (97.1k MWh) alcanza los 31k MWh, una brecha del 24% que no existia a las 00h.
- Este efecto de amplificacion diurna es la firma del consumo industrial y de servicios, que se concentra en horas centrales y desaparece los domingos.
- El sabado presenta perfil intermedio asimetrico: el bloque de 06h cae abruptamente (89.8k MWh vs aprox 111k los laborables) mientras el de 12h recupera parcialmente (105.4k MWh), reflejo de actividad comercial que arranca mas tarde y con menor intensidad.

El efecto del dia de semana depende del bloque horario. Esto justifica incluir hora_x_finde como feature de ingenieria, ya que un modelo con hora y dia_semana independientes no captura que el perfil horario del domingo es cualitativamente distinto al del miercoles.

### <font color='#C0392B'><b>Mapas</b></font>

In [ ]:
with open('/home/app/src/data/BARCELONA.geojson', 'r') as f:
    geojson_bcn = json.load(f)

MAPA_BASE = dict(
    mapbox_style='carto-positron',
    zoom=11,
    center={'lat': 41.3874, 'lon': 2.1686},
    opacity=0.85,
)

LAYOUT_CLARO = dict(
    title_font_size=16,
    title_font_color='#1B3A5C',
    paper_bgcolor='white',
    margin={'r': 0, 't': 50, 'l': 0, 'b': 0},
    height=700, width=1000,
)

def colorbar_claro(texto):
    return dict(title=dict(text=texto, font=dict(color='#1B3A5C')),
                thickness=15, len=0.6, tickfont=dict(color='#1B3A5C'))

# Consumo medio — YlOrRd
consumo_anio = (
    df.filter(pl.col('nombre_postal').is_not_null() & pl.col('anio').is_not_null())
    .group_by(['cod_postal', 'nombre_postal', 'anio'])
    .agg(pl.col('mwh_total').mean().round(1).alias('mwh_medio'))
    .sort(['cod_postal', 'anio'])
    .with_columns(pl.col('anio').cast(pl.Utf8))
    .to_pandas()
)

fig = px.choropleth_mapbox(
    consumo_anio, geojson=geojson_bcn,
    locations='cod_postal', featureidkey='properties.COD_POSTAL',
    color='mwh_medio', animation_frame='anio',
    color_continuous_scale='YlOrRd',
    hover_name='nombre_postal',
    hover_data={'cod_postal': True, 'mwh_medio': ':.0f'},
    labels={'mwh_medio': 'MWh medio'},
    title='Consumo medio por codigo postal (MWh por bloque 6h)',
    range_color=[consumo_anio['mwh_medio'].min(),
                 consumo_anio['mwh_medio'].quantile(0.95)],
    **MAPA_BASE
)
fig.update_layout(**LAYOUT_CLARO, coloraxis_colorbar=colorbar_claro('MWh medio'))
fig.show()

# LST MODIS y Temperatura Meteocat — mismo rango de color RdBu_r
lst_cp = (
    df.filter(pl.col('nombre_postal').is_not_null() &
              pl.col('lst_celsius').is_not_null() &
              pl.col('anio').is_not_null())
    .group_by(['cod_postal', 'nombre_postal', 'anio'])
    .agg(pl.col('lst_celsius').mean().round(2).alias('lst_media'))
    .sort(['cod_postal', 'anio'])
    .with_columns(pl.col('anio').cast(pl.Utf8))
    .to_pandas()
)

temp_cp = (
    df.filter(pl.col('nombre_postal').is_not_null() &
              pl.col('anio').is_not_null() &
              pl.col('temp_mean').is_not_null())
    .group_by(['cod_postal', 'nombre_postal', 'anio'])
    .agg(pl.col('temp_mean').mean().round(2).alias('temp_media'))
    .sort(['cod_postal', 'anio'])
    .with_columns(pl.col('anio').cast(pl.Utf8))
    .to_pandas()
)

# Rango compartido para ambos mapas de temperatura
temp_min_global = min(temp_cp['temp_media'].min(),
                      lst_cp['lst_media'].quantile(0.10))
temp_max_global = max(temp_cp['temp_media'].quantile(0.95),
                      lst_cp['lst_media'].quantile(0.95))

fig_lst = px.choropleth_mapbox(
    lst_cp, geojson=geojson_bcn,
    locations='cod_postal', featureidkey='properties.COD_POSTAL',
    color='lst_media', animation_frame='anio',
    color_continuous_scale='RdBu_r',
    hover_name='nombre_postal',
    hover_data={'lst_media': ':.2f', 'anio': True},
    labels={'lst_media': 'LST media (°C)'},
    title='Temperatura superficial media LST MODIS por codigo postal',
    range_color=[temp_min_global, temp_max_global],
    **MAPA_BASE
)
fig_lst.update_layout(**LAYOUT_CLARO,
                      coloraxis_colorbar=colorbar_claro('LST media (°C)'))
fig_lst.show()

fig_temp = px.choropleth_mapbox(
    temp_cp, geojson=geojson_bcn,
    locations='cod_postal', featureidkey='properties.COD_POSTAL',
    color='temp_media', animation_frame='anio',
    color_continuous_scale='RdBu_r',
    hover_name='nombre_postal',
    hover_data={'temp_media': ':.2f', 'anio': True},
    labels={'temp_media': 'Temp media (°C)'},
    title='Temperatura media Meteocat por codigo postal',
    range_color=[temp_min_global, temp_max_global],
    **MAPA_BASE
)
fig_temp.update_layout(**LAYOUT_CLARO,
                       coloraxis_colorbar=colorbar_claro('Temp media (°C)'))
fig_temp.show()

### <font color='#C0392B'><b>Tests estadísticos Kruskal-Wallis y Mann-Whitney</b></font>


In [ ]:
VARS_KW = ['hora', 'dia_semana', 'mes', 'anio']

print(f"{'variable':<15} {'H_stat':>12} {'p_value':>14} {'sig':>5}")
print('-' * 50)

for var in VARS_KW:
    grupos = [
        df.filter(pl.col(var) == v)['mwh_total'].drop_nulls().to_numpy()
        for v in sorted(df[var].drop_nulls().unique().to_list())
    ]
    h_stat, p_val = kruskal(*grupos)
    sig = '✓' if p_val < 0.05 else '✗'
    print(f"{var:<15} {h_stat:>12.3f} {p_val:>14.4e} {sig:>5}")

print(f"\n{'variable':<15} {'mediana_0':>12} {'mediana_1':>12} {'p_value':>14} {'sig':>5}")
print('-' * 60)

for var in ['es_finde', 'es_festivo']:
    g0 = df.filter(pl.col(var) == 0)['mwh_total'].drop_nulls().to_numpy()
    g1 = df.filter(pl.col(var) == 1)['mwh_total'].drop_nulls().to_numpy()
    stat, p_val = mannwhitneyu(g0, g1, alternative='two-sided')
    sig = '✓' if p_val < 0.05 else '✗'
    print(f"{var:<15} {np.median(g0):>12.1f} {np.median(g1):>12.1f} {p_val:>14.4e} {sig:>5}")

# Kruskal-Wallis: cod_postal
grupos_cp = [
    df.filter(pl.col('cod_postal') == cp)['mwh_total'].drop_nulls().to_numpy()
    for cp in sorted(df['cod_postal'].unique().to_list())
]
h_stat, p_val = kruskal(*grupos_cp)
print(f"\n{'variable':<15} {'H_stat':>12} {'p_value':>14} {'sig':>5}")
print('-' * 50)
print(f"{'cod_postal':<15} {h_stat:>12.3f} {p_val:>14.4e} {'✓' if p_val < 0.05 else '✗':>5}")

> Tests de hipotesis — variables categoricas (Kruskal-Wallis y Mann-Whitney)

- Todas las variables son estadisticamente significativas (p < 0.05).
- cod_postal es el factor mas discriminante del dataset (H_stat=279.152), por
  encima de cualquier variable temporal.
- hora es la variable temporal con mayor poder discriminante (H_stat=48.958),
  seguida de mes (6.355) y dia_semana (5.946).

---
# <font color='#1B3A5C'>  **Análisis de Correlación Spearman** </font>

In [ ]:
VARS_CORR = ['mwh_total', 'temp_mean', 'temp_max', 'temp_min',
             'humedad_mean', 'viento_mean', 'precipitacion_sum',
             'irradiancia_mean', 'lst_celsius']

df_corr = df.select(VARS_CORR).drop_nulls().to_pandas()
corr_matrix = df_corr.corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    ax=ax,
    linewidths=0.5
)
ax.set_title('Correlación de Spearman — variables numéricas', fontweight='bold')
plt.tight_layout()
plt.show()

> Correlacion de Spearman variables numericas

- La correlacion de mwh_total con las variables climaticas es baja pero significativa. 
    - temp_mean (0.16), irradiancia_mean (0.16) y lst_celsius (0.07). La relacion en U con la temperatura explica el valor bajo ya que Spearman subestima relaciones no lineales.
    - temp_mean, temp_max y temp_min correlacionan entre si a 0.95-0.99, colinealidad extrema. Solo temp_mean entra al modelo, las otras dos se descartan.
- lst_celsius correlaciona con temp_mean a 0.87 pero no es redundante. LST mide
  temperatura superficial con variacion espacial por barrio (UHI), mientras
  temp_mean es igual para todos los CPs de la misma estacion Meteocat.
- humedad_mean e irradiancia_mean aportan dimensiones independientes y entran al modelo.
- viento_mean y precipitacion_sum muestran correlacion cercana a 0 con mwh_total.
  Su aportacion se revisara via SHAP tras el modelado.

---
# <font color='#1B3A5C'>  **PCA** </font>

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

VARS_NUM = [
    'temp_mean',
    'humedad_mean', 'viento_mean', 'precipitacion_sum',
    'irradiancia_mean', 'lst_celsius'
]

# Preparar matriz sin nulos
df_pca_raw = df.select(VARS_NUM).drop_nulls().to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca_raw)

pca = PCA(n_components=len(VARS_NUM))
pca.fit(X_scaled)

# Varianza explicada
var_exp_ratio = pca.explained_variance_ratio_
var_acum      = np.cumsum(var_exp_ratio)

print(f"{'PC':<6} {'Var explicada':>15} {'Var acumulada':>15}")
print('-' * 40)
for i, (v, va) in enumerate(zip(var_exp_ratio, var_acum)):
    print(f"PC{i+1:<4} {v:>14.4f} {va:>14.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Barras de fondo por componente
for i, v in enumerate(var_exp_ratio):
    ax.bar(i + 1, v, color=C1, alpha=0.15, width=0.6, zorder=1)

# Linea acumulada
ax.plot(range(1, len(var_acum) + 1), var_acum,
        color=C1, linewidth=2.5, zorder=3)
ax.scatter(range(1, len(var_acum) + 1), var_acum,
           color=C1, s=80, zorder=4,
           edgecolors='white', linewidths=1.5)

# Etiqueta de valor en cada punto
for i, v in enumerate(var_acum):
    ax.text(i + 1, v + 0.02, f'{v*100:.1f}%',
            ha='center', fontsize=8,
            color=C1, fontweight='bold')

# Linea de referencia 90%
ax.axhline(0.90, color='gray', linewidth=1,
           linestyle='--', alpha=0.6)
ax.text(len(var_acum) * 0.65, 0.91, '90% varianza',
        fontsize=8, color='gray')

# PC4 marcado como punto de corte
ax.axvline(4, color=C3, linewidth=1,
           linestyle='--', alpha=0.6)
ax.text(4.1, 0.3, '4 PCs = 89.6%',
        fontsize=8, color=C3, fontweight='bold')

ax.set_title('Grafico de Codo — Varianza Acumulada PCA',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Componente Principal', fontsize=9)
ax.set_ylabel('Varianza Explicada Acumulada', fontsize=9)
ax.set_ylim(0, 1.08)
ax.set_xticks(range(1, len(var_acum) + 1))
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%')
)

plt.tight_layout()
plt.show()

In [ ]:
# Loadings
loadings_df = pd.DataFrame(
    pca.components_.T,
    index=VARS_NUM,
    columns=[f'PC{i+1}' for i in range(len(VARS_NUM))]
).round(4)

print("Loadings:")
print(loadings_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# RdBu_r consistente con los mapas de temperatura
sns.heatmap(
    loadings_df.iloc[:, :4],
    annot=True, fmt='.2f',
    cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=2, linecolor='white',
    annot_kws={'size': 10, 'fontweight': 'bold'},
    ax=ax,
    cbar_kws={'label': 'Loading', 'shrink': 0.8}
)

ax.set_title('Loadings PCA — PC1 a PC4',
             fontweight='bold', fontsize=12, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(fontsize=10, fontweight='bold')
plt.yticks(fontsize=9, rotation=0)
plt.tight_layout()
plt.show()

> PCA variables meteorologicas

- 4 componentes acumulan el 89.6% de la varianza, confirmando que las 6 variables
  aportan informacion diferenciada sin redundancia problematica.
- PC1 (36.2%) eje termico-solar: temp_mean (0.62), lst_celsius (0.57) e
  irradiancia_mean (0.47). Agrupa dias calidos y soleados.
- PC2 (23.6%) eje tiempo adverso: humedad alta (0.58) vs viento fuerte (-0.57)
  e irradiancia baja (-0.37). Captura condiciones meteorologicas desfavorables.
- PC3 (16.9%) eje precipitacion: precipitacion_sum domina con loading 0.94,
  dimension completamente independiente del resto.
- PC4 (12.8%) eje viento humedo: viento (0.73) y humedad (0.55) como combinacion
  de condiciones de brisa maritima.
- El PCA es confirmatorio, no transformador: XGBoost y LSTM gestionan la
  colinealidad de forma nativa y reducir a componentes eliminaria la
  interpretabilidad de SHAP. Las variables originales entran al modelo.

In [ ]:
elapsed = time.time() - start_time
print(f"Tiempo de ejecución del notebook: {elapsed/60:.1f} min ({elapsed:.0f} seg)")

---
# <font color='#1B3A5C'>  **Conclusiones Globales** </font>

> Dataset

- 424.368 registros x 30 columnas (2019-2025, bloques de 6h, 42 codigos postales).
- Sin nulos en mwh_total ni duplicados en datetime + cod_postal tras la limpieza.

> Calidad de datos

- 220 bloques faltantes (08011 ago 2025 + 19-ago global) imputados con mediana historica.
- 378 registros con mwh_total = 0 en 3 fechas de fallo OpenData BCN imputados con
  mediana historica.
- 520 outliers erroneos en 6 CPs (acumulaciones de reporte) imputados con mediana
  2022-2024.
- Estacion AN reasignada a X2 con datos reales de Meteocat.
- Nulos meteorologicos restantes (viento, irradiancia, temp/humedad X2 2024-2025)
  son estructurales y se gestionan en feature engineering.

> Estructura temporal de mwh_total

- Intradiario: pico a las 12h (120k MWh) y valle a las 00h (71k MWh), diferencia
  del 69%. Variable mas discriminante del grupo temporal (Kruskal-Wallis H=48.958).
- Semanal: laborables aprox 106k MWh vs domingo 86k MWh (aprox 19% menos).
  Festivos reducen un 15% adicional respecto a laborables.
- Anual: doble pico verano-invierno con valle en abril-mayo. Caida COVID 2020
  (aprox 12%) y recuperacion progresiva hasta 2023.
- Todos los tests Kruskal-Wallis son significativos (p < 0.001).

> Heterogeneidad espacial

- Rango de 8x entre barrios: desde 29.7k MWh (08033 Vallbona) hasta 233.6k MWh
  (08040 Zona Franca). cod_postal es el factor mas discriminante del dataset
  (Kruskal-Wallis H=279.152, por encima de cualquier variable temporal).
- Barrios industriales (08040, 08038, 08030) lideran el consumo con CV bajo
  (aprox 22-28%), los mas predecibles.
- Barrios de servicios (08011, 08036, 08007-09) presentan CV > 50%, los mas
  dificiles de predecir y donde el modelo concentrara los errores mayores.

> Variables predictivas

- Alta senal: hora, dia_semana, mes, es_finde, es_festivo, cod_postal, confirmadas
  por Kruskal-Wallis y Mann-Whitney (p < 0.001).
- Senal moderada: temp_mean, lst_celsius, humedad_mean, irradiancia_mean,
  significativas por Mann-Whitney con relacion no lineal (en J/U) con mwh_total.
- Senal baja: viento_mean y precipitacion_sum, significativas estadisticamente
  pero con diferencia practica minima. Su aportacion se revisara via SHAP.
- Excluidas por data leakage: mwh_industria, mwh_residencial, mwh_servicios,
  derivadas directamente del target.
- lst_celsius complementaria a temp_mean: captura variacion espacial intraurbana
  por Urban Heat Island. Spearman 0.87 entre ambas pero señales distintas por
  naturaleza (aire vs superficie).

> Estacionariedad

- ADF confirma estacionariedad en media (p < 0.05) en los 5 casos analizados
  (global + 4 CPs representativos).
- KPSS rechaza estacionariedad en varianza: cambios estructurales visibles en STL
  (COVID 2020, recuperacion gradual, mayor variabilidad industrial en 2024-2025).
- XGBoost y LSTM/GRU son robustos a esta condicion. SARIMA requerira diferenciacion
  estacional para cumplir sus supuestos formales.

> Lags identificados

- lag_4 (24h, PACF 0.75): contribucion directa mas fuerte de la serie.
- lag_28 (7 dias, ACF 0.87): el mas parecido de toda la serie, captura patron semanal.
- lag_1 (6h, PACF 0.37): contribucion directa moderada del bloque inmediato anterior.
- rolling_mean_7d: captura tendencia reciente sin depender de un solo bloque puntual.

> PCA confirmatorio

- 4 componentes explican el 89.6% de la varianza meteorologica: PC1 eje termico-solar
  (temp + LST + irradiancia), PC2 tiempo adverso, PC3 precipitacion, PC4 viento humedo.
- Las variables originales entran al modelo para mantener interpretabilidad SHAP.
  El PCA es confirmatorio, no transformador.

> Tendencia termica (2019-2025)

- temp_mean sube +0.31 C/año y humedad_mean desciende -0.31%/año de forma simetrica,
  indicativo de calentamiento urbano progresivo en el periodo analizado.
- Con 7 puntos anuales la tendencia es indicativa, no concluyente estadisticamente.
  Se documenta como contexto para la interpretacion del modelo.

> Limitaciones documentadas

- MODIS MOD11A1 (Terra, aprox 10:30h local): mismo valor LST asignado a los 4 bloques
  del dia. MYD11A1 (Aqua, aprox 13:30h) capturia mejor el pico UHI. Documentado
  como linea futura.
- Cobertura LST en invierno: febrero-marzo con solo 32-34% de registros validos por
  nubosidad. Los valores disponibles corresponden a dias despejados y frios,
  infraestimando la LST media real.
- CP 08010 Pl. Catalunya y 08008-08009 Dreta de l'Eixample: consumo bajo por
  superficie reducida del CP, no por baja actividad economica.

---
# <font color='#1B3A5C'>  **Feature Engineering** </font>

> Lags temporales

- lag_1 (6h antes): contribucion directa confirmada por PACF (0.37).
- lag_4 (24h antes): el mas importante, PACF 0.75. Mismo bloque del dia anterior.
- lag_28 (7 dias antes): ACF 0.87, captura el patron semanal completo.
- rolling_mean_7d: promedio de los ultimos 7 dias, captura tendencia reciente.
- lag_56 y lag_84 (2 y 3 semanas): correlacion existe pero decrece. lag_28
  ya captura el patron semanal. Se evaluan tras modelado via SHAP.

> Variables de interaccion

- hora_x_finde = hora * es_finde: captura que el perfil horario del domingo
  es cualitativamente distinto al del miercoles. Solo XGBoost y LightGBM.
- is_covid = 1 si anio == 2020: controla el cambio estructural por pandemia.

> Variables climaticas derivadas

- HDD = max(0, 15 - temp_mean): grados de calefaccion [Kuru & Calis 2019].
- CDD = max(0, temp_mean - 22): grados de refrigeracion [Kuru & Calis 2019].
- lst_nublado = 1 si lst_celsius es nulo: flag de fiabilidad de MODIS.
- precipitacion_llueve = 1 si precipitacion_sum > 0: binarizacion confirmada
  en EDA (ambas medianas en 0, informacion util solo en si llueve o no).

> Imputacion meteorologica X2

- temp X2 2025: imputar desde X4 con factor de correccion historico
  ratio X2/X4 (2019-2023).
- viento e irradiancia X2: siempre desde X4 (X2 nunca tuvo sensores).
- lst_celsius sin cobertura: MODIS como respaldo espacial.
- irradiancia_mean: topar a 0 los valores negativos de sensor.

> Vecinos geograficos (deprioritizado)

- lag1_vecinos: consumo promedio de CPs colindantes en t-1, calculado con
  shapely sobre BARCELONA.geojson. Se descarta si hay restriccion de tiempo
  y se documenta como linea futura.